In [88]:
""" This is the analysis that 
IRASA analysis time-locked to SR_REC_WORD events.
Experiments: DBOY1, EFRCourierReadOnly
Regions: LHP, RHP only
Window: -750 to 0 ms pre-recall
Output: one row per subject × session × recall_event × region × frequency

Parallelized across subjects using joblib (n_jobs=-1).
recalled = 1 if itemno > 0, else 0 (intrusions / incorrect).
"""

import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from joblib import Parallel, delayed

# ============================================================================
# IRASA IMPORTS
# ============================================================================
try:
    from irasa.IRASA import IRASA, SSL_transform
    IRASA_AVAILABLE = True
    print("✓ IRASA imported successfully")
except ImportError as e:
    IRASA_AVAILABLE = False
    print(f"✗ IRASA not available: {e}")
    def SSL_transform(x):
        return np.sign(x) * np.log10(1 + np.abs(x))

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly']

# IRASA parameters
IRASA_HSET  = np.arange(1.1, 1.9, 0.05)
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)

# Hippocampus only
REGIONS = ['LHP', 'RHP']

# Epoch: load buffer around event, then trim to analysis window
# 500 ms edge buffer on each side to absorb IRASA artefacts
EPOCH_START_MS = -4000   # ms relative to SR_REC_WORD
EPOCH_END_MS   =  2000   # ms relative to SR_REC_WORD
WIN_START_MS   =  -750   # analysis window start (ms)
WIN_END_MS     =     0   # analysis window end   (ms)

# Event type to lock to
POINTING_EVENT_TYPE = 'SR_REC_WORD'

OUTPUT_DIR = './RS_Recall'

N_JOBS = -1   # use all available CPUs

# ============================================================================
# HELPERS
# ============================================================================

def ssl(x):
    return SSL_transform(x)

def ms_to_sample(ms, epoch_start_ms, sfreq):
    return int(round((ms - epoch_start_ms) / 1000.0 * sfreq))

def define_hp_masks(pairs_df):
    """Return LHP / RHP boolean masks, trying multiple region columns."""
    for col in ['dk.region', 'mni.region', 'ind.region']:
        if col in pairs_df.columns:
            region_col = col
            break
    else:
        candidates = [c for c in pairs_df.columns if 'region' in c.lower()]
        if candidates:
            region_col = candidates[0]
            print(f"    WARNING: using {region_col} for region labels")
        else:
            print("    ERROR: no region column found – returning empty masks")
            return {r: np.zeros(len(pairs_df), dtype=bool) for r in REGIONS}

    masks = {
        'LHP': pairs_df[region_col].str.contains('Left.*Hippocampus',  case=False, na=False, regex=True),
        'RHP': pairs_df[region_col].str.contains('Right.*Hippocampus', case=False, na=False, regex=True),
    }
    return masks

def compute_irasa_power_trial(eeg_1d, sfreq, freqs, hset):
    """IRASA on a single 1-D epoch (one channel, one event)."""
    ir = IRASA(
        sig=eeg_1d.astype(float),
        freqs=freqs,
        samplerate=sfreq,
        hset=hset,
        flag_filter=1,
        flag_detrend=1
    )
    mixed   = ir.mixed
    fractal = ir.fractal
    osc     = mixed - fractal
    return mixed, fractal, osc

def irasa_for_event(eeg_window, ch_indices, sfreq):
    """
    Compute SSL-transformed IRASA across all channels in ch_indices for one
    event epoch (eeg_window shape: n_channels × n_samples).

    Returns
    -------
    frac_ssl_mean : array (n_freqs,)  – mean across channels after SSL
    osc_ssl_mean  : array (n_freqs,)
    n_valid_ch    : int
    """
    frac_list = []
    osc_list  = []

    for ch_idx in ch_indices:
        sig = eeg_window[ch_idx, :]
        try:
            _, fractal, osc = compute_irasa_power_trial(sig, sfreq, IRASA_FREQS, IRASA_HSET)
            frac_list.append(ssl(fractal))
            osc_list.append(ssl(osc))
        except Exception:
            pass  # skip noisy / bad channels

    if len(frac_list) == 0:
        nan_vec = np.full(len(IRASA_FREQS), np.nan)
        return nan_vec, nan_vec, 0

    frac_ssl_mean = np.mean(frac_list, axis=0)
    osc_ssl_mean  = np.mean(osc_list,  axis=0)
    return frac_ssl_mean, osc_ssl_mean, len(frac_list)

# ============================================================================
# PER-SUBJECT WORKER  (called in parallel)
# ============================================================================

def process_subject(subject, exp, exp_df, output_dir):
    """
    Process all sessions for one subject in one experiment.

    Returns a list of row-dicts (may be empty).
    Each worker writes its own per-session CSVs directly to disk so partial
    results are preserved even if the master aggregation fails.
    """
    sub_df   = exp_df.query('subject == @subject')
    sessions = sub_df['session'].unique()

    subject_rows = []
    pairs_df     = None
    region_masks = None

    for session in sessions:
        print(f"  [{exp}] {subject} | session {session}")
        session_rows = []

        # ---- skip already-completed sessions --------------------------------
        fname = os.path.join(output_dir, f'{exp}_{subject}_{session}_RS_Recall_irasa.csv')
        if os.path.exists(fname):
            print(f"    ↩ Already exists, skipping: {fname}")
            try:
                df_existing = pd.read_csv(fname)
                subject_rows.extend(df_existing.to_dict('records'))
            except Exception:
                pass
            continue

        try:
            reader = cml.CMLReader(subject, exp, session=session)

            # Load pairs once per subject (same across sessions)
            if pairs_df is None:
                pairs_df     = reader.load('pairs')
                region_masks = define_hp_masks(pairs_df)
                for reg, msk in region_masks.items():
                    n = msk.sum()
                    print(f"    {reg}: {n} bipolar pairs" + (" ⚠ none!" if n == 0 else ""))

            evs = reader.load('task_events')

            # ---- find SR_REC_WORD events ------------------------------------
            pointing_evs = evs[evs['type'] == POINTING_EVENT_TYPE].copy()

            if len(pointing_evs) == 0:
                alt = evs[evs['type'].str.upper() == POINTING_EVENT_TYPE.upper()]
                if len(alt) > 0:
                    pointing_evs = alt.copy()
                    print(f"    Found {len(pointing_evs)} {pointing_evs['type'].iloc[0]} events (case variant)")
                else:
                    available = evs['type'].unique().tolist()
                    print(f"    ⚠ No {POINTING_EVENT_TYPE} events. Available types: {available}")
                    continue

            print(f"    Found {len(pointing_evs)} {POINTING_EVENT_TYPE} events")

            # ---- load EEG for all events at once ----------------------------
            try:
                eeg_container = reader.load_eeg(
                    pointing_evs,
                    EPOCH_START_MS,
                    EPOCH_END_MS,
                    scheme=pairs_df
                )
            except Exception as e:
                print(f"    ✗ EEG load failed: {e}")
                continue

            eeg_data = eeg_container.data          # (n_events, n_channels, n_samples)
            sfreq    = float(eeg_container.samplerate)

            # Sample indices for the analysis window within the loaded epoch
            s0 = ms_to_sample(WIN_START_MS, EPOCH_START_MS, sfreq)
            s1 = ms_to_sample(WIN_END_MS,   EPOCH_START_MS, sfreq)

            print(f"    sfreq={sfreq:.0f} Hz | "
                  f"epoch samples: {eeg_data.shape[2]} | "
                  f"analysis window: [{s0}, {s1})")

            # ---- IRASA per event × region × frequency ----------------------
            pointing_evs_reset = pointing_evs.reset_index(drop=True)

            for ev_idx in range(len(pointing_evs_reset)):
                ev_row = pointing_evs_reset.iloc[ev_idx]

                trial       = ev_row.get('trial',          np.nan)
                eegoffset   = ev_row.get('eegoffset',      np.nan)
                item        = ev_row.get('item',           '')
                store_x     = ev_row.get('storeX',         np.nan)
                store_z     = ev_row.get('storeZ',         np.nan)
                inside_stim = ev_row.get('inside_stimuli', np.nan)

                # recalled flag: 1 if valid recall (itemno > 0), 0 if intrusion
                itemno   = ev_row.get('itemno', np.nan)
                recalled = 1 if (not (isinstance(itemno, float) and np.isnan(itemno)) and itemno > 0) else 0

                eeg_epoch = eeg_data[ev_idx, :, s0:s1]  # (n_ch, n_win_samples)

                for region in REGIONS:
                    ch_indices = np.where(region_masks[region])[0]

                    frac_ssl_mean, osc_ssl_mean, n_valid_ch = irasa_for_event(
                        eeg_epoch, ch_indices, sfreq
                    )

                    for freq_idx, freq_hz in enumerate(IRASA_FREQS):
                        session_rows.append({
                            'experiment':   exp,
                            'subject':      subject,
                            'session':      session,
                            'event_idx':    ev_idx,
                            'trial':        trial,
                            'eegoffset':    eegoffset,
                            'item':         item,
                            'storeX':       store_x,
                            'storeZ':       store_z,
                            'inside_stim':  inside_stim,
                            'itemno':       itemno,
                            'recalled':     recalled,
                            'region':       region,
                            'n_channels':   n_valid_ch,
                            'freq_hz':      freq_hz,
                            'freq_idx':     freq_idx,
                            'frac_ssl':     frac_ssl_mean[freq_idx],
                            'osc_ssl':      osc_ssl_mean[freq_idx],
                        })

            # ---- save session CSV ------------------------------------------
            if session_rows:
                sess_df = pd.DataFrame(session_rows)
                sess_df.to_csv(fname, index=False)
                print(f"    ✓ Saved {len(session_rows):,} rows → {fname}")
                subject_rows.extend(session_rows)
            else:
                print(f"    ⚠ No rows produced for session {session}")

        except Exception as e:
            import traceback
            print(f"  ✗ [{exp}] {subject} session {session} FAILED: {e}")
            traceback.print_exc()
            continue

    return subject_rows

# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    whole_df = cml.CMLReader.get_data_index()

    all_rows = []

    for exp in EXPERIMENTS:
        exp_df   = whole_df.query('experiment == @exp')
        subjects = exp_df['subject'].unique()

        print(f"\n{'='*80}")
        print(f"Experiment: {exp}  |  {len(subjects)} subjects  |  n_jobs={N_JOBS}")
        print(f"{'='*80}")

        results = Parallel(n_jobs=N_JOBS, backend='loky', verbose=10)(
            delayed(process_subject)(subject, exp, exp_df, OUTPUT_DIR)
            for subject in subjects
        )

        for subj_rows in results:
            all_rows.extend(subj_rows)

    # ============================================================================
    # MASTER CSV
    # ============================================================================
    print("\n" + "="*80)
    print("ALL DONE")
    print("="*80)

    if all_rows:
        master_df  = pd.DataFrame(all_rows)
        master_csv = os.path.join(OUTPUT_DIR, 'ALL_SUBJECTS_RS_Recall_irasa.csv')
        master_df.to_csv(master_csv, index=False)

        print(f"\n✓ Master CSV: {master_csv}")
        print(f"  Rows        : {len(master_df):,}")
        print(f"  Experiments : {master_df['experiment'].unique().tolist()}")
        print(f"  Subjects    : {master_df['subject'].nunique()}")
        print(f"  Sessions    : {master_df['session'].nunique()}")
        print(f"  Events      : {master_df.groupby(['experiment','subject','session','event_idx']).ngroups:,}")
        print(f"  Recalled==1 : {(master_df['recalled']==1).sum():,}")
        print(f"  Recalled==0 : {(master_df['recalled']==0).sum():,}")
        print(f"  Regions     : {sorted(master_df['region'].unique())}")
        print(f"  Frequencies : {master_df['freq_hz'].nunique()}")
        print(f"\nSample rows:")
        print(master_df.head(10).to_string(index=False))
    else:
        print("\n⚠ No data collected – check event type labels above.")

✓ IRASA imported successfully

Experiment: DBOY1  |  44 subjects  |  n_jobs=-1


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:   37.6s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:  3.8min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  6.1min
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:  6.3min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed: 10.9min
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed: 12.6min
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed: 14.2min
[Parallel(n_jobs=-1)]: Done  44 out of  44 | elapsed: 16.2min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.



Experiment: EFRCourierReadOnly  |  8 subjects  |  n_jobs=-1


[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    4.0s
[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:   31.8s
[Parallel(n_jobs=-1)]: Done   6 out of   8 | elapsed:  2.4min remaining:   47.6s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:  4.0min remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   8 out of   8 | elapsed:  4.0min finished



ALL DONE

✓ Master CSV: ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa.csv
  Rows        : 41,436
  Experiments : ['DBOY1', 'EFRCourierReadOnly']
  Subjects    : 43
  Sessions    : 8
  Events      : 1,151
  Recalled==1 : 38,484
  Recalled==0 : 2,952
  Regions     : ['LHP', 'RHP']
  Frequencies : 18

Sample rows:
experiment subject  session  event_idx  trial  eegoffset           item  storeX  storeZ  inside_stim  itemno  recalled region  n_channels  freq_hz  freq_idx  frac_ssl  osc_ssl
     DBOY1  R1494D        0          0   -999    4539622 HARDWARE STORE  -999.0  -999.0          NaN      11         1    LHP           0     3.00         0       NaN      NaN
     DBOY1  R1494D        0          0   -999    4539622 HARDWARE STORE  -999.0  -999.0          NaN      11         1    LHP           0     3.74         1       NaN      NaN
     DBOY1  R1494D        0          0   -999    4539622 HARDWARE STORE  -999.0  -999.0          NaN      11         1    LHP           0     4.67         2       

In [89]:
"""
Post-processing: average IRASA features across events AND sessions
for each subject × experiment × region × frequency.
Input : ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa.csv   (trial-level master CSV)
Output: ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa_subj_avg.csv

Also produces separate averages for recalled=1 and recalled=0 events.
"""
import os
import pandas as pd
import numpy as np

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_CSV  = './RS_Recall/ALL_SUBJECTS_RS_Recall_irasa.csv'
OUTPUT_CSV = './RS_Recall/ALL_SUBJECTS_RS_Recall_irasa_subj_avg.csv'
OUTPUT_CSV_BY_RECALLED = './RS_Recall/ALL_SUBJECTS_RS_Recall_irasa_subj_avg_by_recalled.csv'

# ============================================================================
# LOAD
# ============================================================================
print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Rows loaded  : {len(df):,}")
print(f"  Subjects     : {df['subject'].nunique()}")
print(f"  Experiments  : {df['experiment'].unique().tolist()}")
print(f"  Sessions     : {df['session'].nunique()}")
print(f"  Regions      : {sorted(df['region'].unique())}")
print(f"  Frequencies  : {df['freq_hz'].nunique()}")
print(f"  Recalled==1  : {(df['recalled']==1).sum():,}")
print(f"  Recalled==0  : {(df['recalled']==0).sum():,}")

# ============================================================================
# STEP 1: session-level average (collapsed across events)
# ============================================================================
print("\nStep 1: session-level averages ...")
session_avg = (
    df
    .groupby(
        ['experiment', 'subject', 'session', 'region', 'freq_hz', 'freq_idx'],
        sort=True, observed=True
    )
    .agg(
        n_events   = ('event_idx',  'nunique'),
        n_channels = ('n_channels', 'mean'),
        frac_ssl   = ('frac_ssl',   'mean'),
        osc_ssl    = ('osc_ssl',    'mean'),
    )
    .reset_index()
)
print(f"  Session-level rows: {len(session_avg):,}")

# ============================================================================
# STEP 2: subject-level average (averaging over sessions equally)
# ============================================================================
print("Step 2: subject-level averages (averaging over sessions) ...")
subj_avg = (
    session_avg
    .groupby(
        ['experiment', 'subject', 'region', 'freq_hz', 'freq_idx'],
        sort=True, observed=True
    )
    .agg(
        n_sessions  = ('session',    'nunique'),
        n_events    = ('n_events',   'sum'),
        n_channels  = ('n_channels', 'mean'),
        frac_ssl    = ('frac_ssl',   'mean'),
        osc_ssl     = ('osc_ssl',    'mean'),
    )
    .reset_index()
)

# ============================================================================
# SAVE — collapsed average
# ============================================================================
os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
subj_avg.to_csv(OUTPUT_CSV, index=False)
print(f"\n✓ Saved subject-average CSV : {OUTPUT_CSV}")
print(f"  Rows : {len(subj_avg):,}  "
      f"(= {subj_avg['subject'].nunique()} subj × "
      f"{subj_avg['region'].nunique()} regions × "
      f"{subj_avg['freq_hz'].nunique()} freqs)")

# ============================================================================
# STEP 3: same pipeline split by recalled (1 vs 0)
# ============================================================================
print("\nStep 3: session-level averages split by recalled ...")
session_avg_rec = (
    df
    .groupby(
        ['experiment', 'subject', 'session', 'recalled', 'region', 'freq_hz', 'freq_idx'],
        sort=True, observed=True
    )
    .agg(
        n_events   = ('event_idx',  'nunique'),
        n_channels = ('n_channels', 'mean'),
        frac_ssl   = ('frac_ssl',   'mean'),
        osc_ssl    = ('osc_ssl',    'mean'),
    )
    .reset_index()
)

print("Step 4: subject-level averages split by recalled ...")
subj_avg_rec = (
    session_avg_rec
    .groupby(
        ['experiment', 'subject', 'recalled', 'region', 'freq_hz', 'freq_idx'],
        sort=True, observed=True
    )
    .agg(
        n_sessions  = ('session',    'nunique'),
        n_events    = ('n_events',   'sum'),
        n_channels  = ('n_channels', 'mean'),
        frac_ssl    = ('frac_ssl',   'mean'),
        osc_ssl     = ('osc_ssl',    'mean'),
    )
    .reset_index()
)

# ============================================================================
# SAVE — by-recalled average
# ============================================================================
subj_avg_rec.to_csv(OUTPUT_CSV_BY_RECALLED, index=False)
print(f"\n✓ Saved by-recalled CSV     : {OUTPUT_CSV_BY_RECALLED}")
print(f"  Rows : {len(subj_avg_rec):,}  "
      f"(= {subj_avg_rec['subject'].nunique()} subj × "
      f"2 recalled conditions × "
      f"{subj_avg_rec['region'].nunique()} regions × "
      f"{subj_avg_rec['freq_hz'].nunique()} freqs)")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*60)
print("DONE")
print("="*60)
print(f"\nCollapsed average sample:")
print(subj_avg.head(6).to_string(index=False))
print(f"\nBy-recalled sample:")
print(subj_avg_rec.head(12).to_string(index=False))

Loading ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa.csv ...
  Rows loaded  : 41,436
  Subjects     : 43
  Experiments  : ['DBOY1', 'EFRCourierReadOnly']
  Sessions     : 8
  Regions      : ['LHP', 'RHP']
  Frequencies  : 18
  Recalled==1  : 38,484
  Recalled==0  : 2,952

Step 1: session-level averages ...
  Session-level rows: 3,240
Step 2: subject-level averages (averaging over sessions) ...

✓ Saved subject-average CSV : ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa_subj_avg.csv
  Rows : 1,548  (= 43 subj × 2 regions × 18 freqs)

Step 3: session-level averages split by recalled ...
Step 4: subject-level averages split by recalled ...

✓ Saved by-recalled CSV     : ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa_subj_avg_by_recalled.csv
  Rows : 2,376  (= 43 subj × 2 recalled conditions × 2 regions × 18 freqs)

DONE

Collapsed average sample:
experiment subject region  freq_hz  freq_idx  n_sessions  n_events  n_channels  frac_ssl  osc_ssl
     DBOY1  R1494D    LHP     3.00         0           3     

In [92]:
"""
Merge two IRASA pipelines into a single long-format dataframe for comparison.

Sources
-------
A) RS_Recall pipeline  : ./RS_Recall/ALL_SUBJECTS_RS_Recall_irasa.csv
   - Event type        : SR_REC_WORD (time-locked to recall onset)
   - Regions           : LHP, RHP
   - Power cols        : frac_ssl, osc_ssl

B) Extended-ROI pipeline: ./subject_results_irasa_per_freq_extended_rois/ALL_SUBJECTS_irasa_per_freq.csv
   - Event type        : REC_WORD (free-recall retrieval window -750–0 ms)
   - Regions           : LHP, RHP (only; LEC/REC/LPC/RPC dropped)
   - Power cols        : retrieval_frac_ssl, retrieval_osc_ssl

Output schema (one row per event × region × frequency)
-------------------------------------------------------
subject, session, trial, item, region, freq_hz, freq_idx,
source,           # 'RS_Recall' or 'REC_WORD_retrieval'
recalled,         # 1 / 0
frac_ssl,         # harmonised power column
osc_ssl,          # harmonised power column
# passthrough cols where available:
SP_clustering_from_prev, SP_clustering_to_next,
T_clustering_from_prev,  T_clustering_to_next,
encoding_stim, recall_stim, inside_stim
"""

import os
import pandas as pd
import numpy as np

# ============================================================================
# CONFIGURATION
# ============================================================================
RS_RECALL_CSV   = './RS_Recall/ALL_SUBJECTS_RS_Recall_irasa.csv'
EXT_ROI_CSV     = 'ALL_SUBJECTS_irasa_clean.csv'
OUTPUT_CSV      = './merged_irasa_rs_recall_vs_retrieval.csv'

KEEP_REGIONS    = ['LHP', 'RHP']

# ============================================================================
# LOAD  A  —  RS_Recall
# ============================================================================
print("Loading RS_Recall CSV ...")
rs = pd.read_csv(RS_RECALL_CSV)
print(f"  Rows      : {len(rs):,}")
print(f"  Subjects  : {rs['subject'].nunique()}")
print(f"  Regions   : {sorted(rs['region'].unique())}")
print(f"  recalled==1: {(rs['recalled']==1).sum():,}  |  recalled==0: {(rs['recalled']==0).sum():,}")

# Keep only LHP / RHP
rs = rs[rs['region'].isin(KEEP_REGIONS)].copy()

# Harmonise column names → unified schema
rs_out = pd.DataFrame({
    'subject'                 : rs['subject'],
    'session'                 : rs['session'],
    'trial'                   : rs['trial'],
    'item'                    : rs['item'],
    'region'                  : rs['region'],
    'freq_hz'                 : rs['freq_hz'],
    'freq_idx'                : rs['freq_idx'],
    'source'                  : 'RS_Recall',
    'recalled'                : rs['recalled'],
    'frac_ssl'                : rs['frac_ssl'],
    'osc_ssl'                 : rs['osc_ssl'],
    # clustering — not available in RS_Recall
    'SP_clustering_from_prev' : np.nan,
    'SP_clustering_to_next'   : np.nan,
    'T_clustering_from_prev'  : np.nan,
    'T_clustering_to_next'    : np.nan,
    # stim info
    'encoding_stim'           : np.nan,
    'recall_stim'             : rs.get('inside_stim', pd.Series(np.nan, index=rs.index)),
    'inside_stim'             : rs.get('inside_stim', pd.Series(np.nan, index=rs.index)),
})

print(f"\n  RS_Recall harmonised rows : {len(rs_out):,}")

# ============================================================================
# LOAD  B  —  Extended-ROI retrieval
# ============================================================================
print("\nLoading extended-ROI CSV ...")
ext = pd.read_csv(EXT_ROI_CSV)
print(f"  Rows      : {len(ext):,}")
print(f"  Subjects  : {ext['subject'].nunique()}")
print(f"  Regions   : {sorted(ext['region'].unique())}")

# Keep only LHP / RHP  +  only retrieval power
ext = ext[ext['region'].isin(KEEP_REGIONS)].copy()

# recalled = 1 for all rows in extended-ROI (only clean correct recalls are stored)
ext_out = pd.DataFrame({
    'subject'                 : ext['subject'],
    'session'                 : ext['session'],
    'trial'                   : ext['trial'],
    'item'                    : ext['recalled_word'],          # rename to common key
    'region'                  : ext['region'],
    'freq_hz'                 : ext['freq_hz'],
    'freq_idx'                : ext['freq_idx'],
    'source'                  : 'REC_WORD_retrieval',
    'recalled'                : 1,                             # only correct recalls
    'frac_ssl'                : ext['retrieval_frac_ssl'],
    'osc_ssl'                 : ext['retrieval_osc_ssl'],
    # clustering scores
    'SP_clustering_from_prev' : ext['SP_clustering_from_prev'],
    'SP_clustering_to_next'   : ext['SP_clustering_to_next'],
    'T_clustering_from_prev'  : ext['T_clustering_from_prev'],
    'T_clustering_to_next'    : ext['T_clustering_to_next'],
    # stim info
    'encoding_stim'           : ext['encoding_stim'],
    'recall_stim'             : ext['recall_stim'],
    'inside_stim'             : np.nan,
})

print(f"  Extended-ROI harmonised rows : {len(ext_out):,}")

# ============================================================================
# CONCATENATE
# ============================================================================
print("\nConcatenating ...")
merged = pd.concat([rs_out, ext_out], ignore_index=True)

# Consistent dtypes
merged['freq_idx'] = merged['freq_idx'].astype(int)
merged['recalled'] = merged['recalled'].astype(int)

# Sort for readability
merged = merged.sort_values(
    ['subject', 'session', 'trial', 'item', 'source', 'region', 'freq_hz']
).reset_index(drop=True)

# ============================================================================
# DIAGNOSTICS
# ============================================================================
print("\n" + "="*60)
print("MERGED DATAFRAME SUMMARY")
print("="*60)
print(f"  Total rows        : {len(merged):,}")
print(f"  Subjects          : {merged['subject'].nunique()}")
print(f"  Sessions          : {merged['session'].nunique()}")
print(f"  Regions           : {sorted(merged['region'].unique())}")
print(f"  Frequencies       : {merged['freq_hz'].nunique()}")
print(f"  Sources           : {merged['source'].unique().tolist()}")
print(f"\n  Rows per source:")
print(merged.groupby('source')[['frac_ssl','osc_ssl']].agg(['count','mean']).round(4).to_string())

print(f"\n  Subjects in RS_Recall only   : "
      f"{set(rs_out['subject'].unique()) - set(ext_out['subject'].unique())}")
print(f"  Subjects in Ext-ROI only     : "
      f"{set(ext_out['subject'].unique()) - set(rs_out['subject'].unique())}")
print(f"  Subjects in BOTH             : "
      f"{len(set(rs_out['subject'].unique()) & set(ext_out['subject'].unique()))}")

# ============================================================================
# SAVE
# ============================================================================
os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False)
print(f"\n✓ Saved merged CSV: {OUTPUT_CSV}")
print(f"\nColumn list:")
print(merged.dtypes.to_string())
print(f"\nSample rows (RS_Recall):")
print(merged[merged['source']=='RS_Recall'].head(6).to_string(index=False))
print(f"\nSample rows (REC_WORD_retrieval):")
print(merged[merged['source']=='REC_WORD_retrieval'].head(6).to_string(index=False))

Loading RS_Recall CSV ...
  Rows      : 41,436
  Subjects  : 43
  Regions   : ['LHP', 'RHP']
  recalled==1: 38,484  |  recalled==0: 2,952

  RS_Recall harmonised rows : 41,436

Loading extended-ROI CSV ...
  Rows      : 27,162
  Subjects  : 26
  Regions   : ['LHP', 'RHP']
  Extended-ROI harmonised rows : 27,162

Concatenating ...

MERGED DATAFRAME SUMMARY
  Total rows        : 68,598
  Subjects          : 45
  Sessions          : 8
  Regions           : ['LHP', 'RHP']
  Frequencies       : 18
  Sources           : ['RS_Recall', 'REC_WORD_retrieval']

  Rows per source:
                   frac_ssl         osc_ssl        
                      count    mean   count    mean
source                                             
REC_WORD_retrieval    27162  2.1449   27162  0.4280
RS_Recall             25776  2.1304   25776  0.4198

  Subjects in RS_Recall only   : {'R1552E', 'R1535E', 'R1526J', 'R1539E', 'R1524T', 'R1520T', 'R1642J', 'R1549E', 'R1663E', 'R1598E', 'R1585E', 'R1653J', 'R1575E',

In [94]:
"""
LME Analysis: RS_Recall vs REC_WORD_retrieval gamma power (>40 Hz)

Model (run separately for frac_ssl and osc_ssl, and for each region):
----------------------------------------------------------------------
    power ~ source * freq_hz + (1 | subject) + (1 | subject:session)

Where:
    - power     : frac_ssl or osc_ssl (within-freq z-scored)
    - source    : 'RS_Recall' vs 'REC_WORD_retrieval' (dummy-coded, RS_Recall = 0)
    - freq_hz   : continuous, gamma only (>40 Hz)
    - subject   : top-level random intercept
    - session   : nested within subject random intercept

Filters:
    - recalled == 1 only (correct recalls)
    - freq_hz > 40 Hz only
    - regions: LHP, RHP (run separately)

Output:
    - CSV of all model results
    - Forest plot per region × power component
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_CSV   = './merged_irasa_rs_recall_vs_retrieval.csv'
OUTPUT_DIR  = './lme_rs_recall_vs_retrieval'
os.makedirs(OUTPUT_DIR, exist_ok=True)

REGIONS          = ['LHP', 'RHP']
POWER_COMPONENTS = ['frac_ssl', 'osc_ssl']
GAMMA_CUTOFF_HZ  = 40.0
REFERENCE_SOURCE = 'RS_Recall'          # will be 0 in dummy coding
OTHER_SOURCE     = 'REC_WORD_retrieval' # will be 1

# ============================================================================
# LOAD & FILTER
# ============================================================================
print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Total rows       : {len(df):,}")

# Correct recalls only
df = df[df['recalled'] == 1].copy()
print(f"  After recalled==1: {len(df):,}")

# Gamma only
df = df[df['freq_hz'] > GAMMA_CUTOFF_HZ].copy()
print(f"  After freq>40 Hz : {len(df):,}")
print(f"  Gamma freqs used : {sorted(df['freq_hz'].unique())}")

# Regions of interest
df = df[df['region'].isin(REGIONS)].copy()

# ---- Keep only subjects who have BOTH sources --------------------------------
subjects_rs  = set(df[df['source'] == REFERENCE_SOURCE]['subject'].unique())
subjects_ext = set(df[df['source'] == OTHER_SOURCE]['subject'].unique())
subjects_both = subjects_rs & subjects_ext

print(f"\n  Subjects with RS_Recall only        : {len(subjects_rs  - subjects_ext)}")
print(f"  Subjects with REC_WORD_retrieval only: {len(subjects_ext - subjects_rs)}")
print(f"  Subjects with BOTH (kept)            : {len(subjects_both)}")

df = df[df['subject'].isin(subjects_both)].copy()
print(f"  Rows after subject filter            : {len(df):,}")

# Dummy-code source: RS_Recall=0, REC_WORD_retrieval=1
df['source_code'] = (df['source'] == OTHER_SOURCE).astype(int)

# Create nested grouping variable for random effects
df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

print(f"\n  Sources          : {df['source'].value_counts().to_dict()}")
print(f"  Subjects         : {df['subject'].nunique()}")
print(f"  subj_sess groups : {df['subj_sess'].nunique()}")

# ============================================================================
# WITHIN-FREQUENCY Z-SCORE (per source × region × freq_hz)
# This removes mean power differences driven purely by frequency, not condition
# ============================================================================
print("\nZ-scoring power within each subject × region × freq_hz ...")

for power_col in POWER_COMPONENTS:
    z_col = f'{power_col}_z'
    df[z_col] = np.nan
    for (subj, region, freq), grp in df.groupby(['subject', 'region', 'freq_hz']):
        mu  = grp[power_col].mean()
        std = grp[power_col].std()
        if std > 0:
            df.loc[grp.index, z_col] = (grp[power_col] - mu) / std
        else:
            df.loc[grp.index, z_col] = 0.0

# ============================================================================
# LME MODELS
# ============================================================================
# Model formula:
#   power_z ~ source_code * freq_hz + (1|subject) + (1|subject:session)
# In statsmodels mixedlm with nested random effects:
#   groups     = subject
#   vc_formula = {'subj_sess': '0 + C(subj_sess)'}  → session within subject

all_results = []

for region in REGIONS:
    for power_col in POWER_COMPONENTS:
        dv      = f'{power_col}_z'
        reg_df  = df[df['region'] == region].dropna(subset=[dv]).copy()

        print(f"\n{'='*65}")
        print(f"  Region: {region}  |  DV: {dv}")
        print(f"  N rows: {len(reg_df):,}  |  N subjects (both phases): {reg_df['subject'].nunique()}")
        print(f"  Sources in this subset: {reg_df['source'].value_counts().to_dict()}")
        print(f"{'='*65}")

        formula = f'{dv} ~ source_code * freq_hz'

        try:
            model = smf.mixedlm(
                formula,
                data      = reg_df,
                groups    = reg_df['subject'],
                vc_formula= {'subj_sess': '0 + C(subj_sess)'}
            )
            result = model.fit(reml=True, method='lbfgs', maxiter=1000)

            print(result.summary())

            # Extract fixed effects
            fe = result.fe_params
            pv = result.pvalues
            ci = result.conf_int()

            for param in fe.index:
                all_results.append({
                    'region'      : region,
                    'power'       : power_col,
                    'parameter'   : param,
                    'beta'        : fe[param],
                    'pval'        : pv[param],
                    'ci_low'      : ci.loc[param, 0],
                    'ci_high'     : ci.loc[param, 1],
                    'n_rows'      : len(reg_df),
                    'n_subjects'  : reg_df['subject'].nunique(),
                    'n_subj_sess' : reg_df['subj_sess'].nunique(),
                    'converged'   : result.converged,
                })

        except Exception as e:
            print(f"  ✗ Model failed: {e}")
            import traceback; traceback.print_exc()
            continue

# ============================================================================
# RESULTS DATAFRAME + FDR CORRECTION
# ============================================================================
results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    print("\n⚠ No results to save.")
else:
    # FDR correction separately for each parameter across region × power combos
    for param in results_df['parameter'].unique():
        mask = results_df['parameter'] == param
        pvals = results_df.loc[mask, 'pval'].values
        if len(pvals) > 1:
            _, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
            results_df.loc[mask, 'pval_fdr'] = pvals_fdr
        else:
            results_df.loc[mask, 'pval_fdr'] = pvals

    results_df['sig_fdr'] = results_df['pval_fdr'] < 0.05
    results_df['sig_raw'] = results_df['pval']      < 0.05

    # Save results
    results_csv = os.path.join(OUTPUT_DIR, 'lme_results_gamma_source_comparison.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"\n✓ Results saved: {results_csv}")
    print(results_df.to_string(index=False))

    # ============================================================================
    # FOREST PLOT  — one panel per region, rows = parameters, colour = power
    # ============================================================================
    params_to_plot = ['source_code', 'freq_hz', 'source_code:freq_hz']
    param_labels   = {
        'source_code'          : 'Source\n(REC_WORD vs RS_Recall)',
        'freq_hz'              : 'Frequency (Hz)',
        'source_code:freq_hz'  : 'Source × Frequency\nInteraction',
    }
    colors = {'frac_ssl': '#4C72B0', 'osc_ssl': '#DD8452'}

    fig, axes = plt.subplots(1, len(REGIONS), figsize=(5 * len(REGIONS), 5), sharey=True)
    if len(REGIONS) == 1:
        axes = [axes]

    for ax, region in zip(axes, REGIONS):
        sub = results_df[results_df['region'] == region].copy()
        sub = sub[sub['parameter'].isin(params_to_plot)]

        y_positions = {}
        spacing = 0.25
        for i, param in enumerate(params_to_plot):
            y_positions[param] = i

        for power_col, offset in zip(POWER_COMPONENTS, [-spacing/2, spacing/2]):
            pwr_sub = sub[sub['power'] == power_col]
            for _, row in pwr_sub.iterrows():
                y   = y_positions[row['parameter']] + offset
                col = colors[power_col]
                marker = '*' if row['sig_fdr'] else ('o' if row['sig_raw'] else 's')
                ax.errorbar(
                    x     = row['beta'],
                    y     = y,
                    xerr  = [[row['beta'] - row['ci_low']],
                              [row['ci_high'] - row['beta']]],
                    fmt   = marker,
                    color = col,
                    markersize = 9 if row['sig_fdr'] else 7,
                    capsize= 4,
                    linewidth= 1.5,
                    label = power_col if row['parameter'] == params_to_plot[0] else ''
                )

        ax.axvline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_yticks(list(y_positions.values()))
        ax.set_yticklabels([param_labels.get(p, p) for p in params_to_plot], fontsize=10)
        ax.set_xlabel('β  (95% CI)', fontsize=11)
        ax.set_title(f'{region}', fontsize=13, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)

    # Legend
    handles = [mpatches.Patch(color=colors[p], label=p) for p in POWER_COMPONENTS]
    handles += [
        plt.Line2D([0],[0], marker='*', color='gray', linestyle='', markersize=9, label='p_FDR < 0.05'),
        plt.Line2D([0],[0], marker='o', color='gray', linestyle='', markersize=7, label='p_raw < 0.05'),
        plt.Line2D([0],[0], marker='s', color='gray', linestyle='', markersize=7, label='n.s.'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, -0.08), frameon=False)

    fig.suptitle(
        'Gamma Power: RS_Recall vs REC_WORD Retrieval\n'
        f'(recalled=1, freq > {GAMMA_CUTOFF_HZ} Hz, LME with nested random effects)',
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'forest_plot_gamma_source_comparison.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Forest plot saved: {plot_path}")

    # ============================================================================
    # DESCRIPTIVE SUMMARY  — mean ± SEM per source × region × freq_hz
    # ============================================================================
    print("\nDescriptive summary (mean ± SEM per source × region):")
    desc = (
        df.groupby(['source', 'region'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
    )
    print(desc.to_string())

    desc_freq = (
        df.groupby(['source', 'region', 'freq_hz'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
        .reset_index()
    )
    desc_csv = os.path.join(OUTPUT_DIR, 'descriptive_gamma_source_by_freq.csv')
    desc_freq.to_csv(desc_csv, index=False)
    print(f"\n✓ Descriptive CSV saved: {desc_csv}")

Loading ./merged_irasa_rs_recall_vs_retrieval.csv ...
  Total rows       : 68,598
  After recalled==1: 65,646
  After freq>40 Hz : 21,882
  Gamma freqs used : [42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

  Subjects with RS_Recall only        : 19
  Subjects with REC_WORD_retrieval only: 2
  Subjects with BOTH (kept)            : 24
  Rows after subject filter            : 17,334

  Sources          : {'RS_Recall': 9048, 'REC_WORD_retrieval': 8286}
  Subjects         : 24
  subj_sess groups : 63

Z-scoring power within each subject × region × freq_hz ...

  Region: LHP  |  DV: frac_ssl_z
  N rows: 8,688  |  N subjects (both phases): 24
  Sources in this subset: {'RS_Recall': 4524, 'REC_WORD_retrieval': 4164}
            Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  frac_ssl_z 
No. Observations:    8688     Method:              REML       
No. Groups:          24       Scale:               0.5877     
Min. group size:     72       Log-Likelihood:

In [96]:
"""
LME Analysis: RS_Recall vs REC_WORD_retrieval gamma power (>40 Hz)

Model (run separately for frac_ssl and osc_ssl, and for each region):
----------------------------------------------------------------------
    power ~ source * freq_hz + (1 | subject) + (1 | subject:session)

Where:
    - power     : frac_ssl or osc_ssl (raw SSL values)
    - source    : 'RS_Recall' vs 'REC_WORD_retrieval' (dummy-coded, RS_Recall = 0)
    - freq_hz   : continuous, gamma only (>40 Hz)
    - subject   : top-level random intercept
    - session   : nested within subject random intercept

Filters:
    - recalled == 1 only (correct recalls)
    - freq_hz > 40 Hz only
    - regions: LHP, RHP (run separately)

Output:
    - CSV of all model results
    - Forest plot per region × power component
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_CSV   = './merged_irasa_rs_recall_vs_retrieval.csv'
OUTPUT_DIR  = './lme_rs_recall_vs_retrieval'
os.makedirs(OUTPUT_DIR, exist_ok=True)

REGIONS          = ['LHP', 'RHP']
POWER_COMPONENTS = ['frac_ssl', 'osc_ssl']
GAMMA_CUTOFF_HZ  = 40.0
REFERENCE_SOURCE = 'RS_Recall'          # will be 0 in dummy coding
OTHER_SOURCE     = 'REC_WORD_retrieval' # will be 1

# ============================================================================
# LOAD & FILTER
# ============================================================================
print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Total rows       : {len(df):,}")

# Correct recalls only
df = df[df['recalled'] == 1].copy()
print(f"  After recalled==1: {len(df):,}")

# Gamma only
df = df[df['freq_hz'] > GAMMA_CUTOFF_HZ].copy()
print(f"  After freq>40 Hz : {len(df):,}")
print(f"  Gamma freqs used : {sorted(df['freq_hz'].unique())}")

# Regions of interest
df = df[df['region'].isin(REGIONS)].copy()

# ---- Keep only subjects who have BOTH sources --------------------------------
subjects_rs  = set(df[df['source'] == REFERENCE_SOURCE]['subject'].unique())
subjects_ext = set(df[df['source'] == OTHER_SOURCE]['subject'].unique())
subjects_both = subjects_rs & subjects_ext

print(f"\n  Subjects with RS_Recall only        : {len(subjects_rs  - subjects_ext)}")
print(f"  Subjects with REC_WORD_retrieval only: {len(subjects_ext - subjects_rs)}")
print(f"  Subjects with BOTH (kept)            : {len(subjects_both)}")

df = df[df['subject'].isin(subjects_both)].copy()
print(f"  Rows after subject filter            : {len(df):,}")

# Dummy-code source: RS_Recall=0, REC_WORD_retrieval=1
df['source_code'] = (df['source'] == OTHER_SOURCE).astype(int)

# Create nested grouping variable for random effects
df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

print(f"\n  Sources          : {df['source'].value_counts().to_dict()}")
print(f"  Subjects         : {df['subject'].nunique()}")
print(f"  subj_sess groups : {df['subj_sess'].nunique()}")

# ============================================================================
# NO Z-SCORING — model raw SSL values directly
# ============================================================================
print("\nUsing raw SSL values (no z-scoring).")

# ============================================================================
# LME MODELS
# ============================================================================
# Model formula:
#   power ~ source_code * freq_hz + (1|subject) + (1|subject:session)
# In statsmodels mixedlm with nested random effects:
#   groups     = subject
#   vc_formula = {'subj_sess': '0 + C(subj_sess)'}  → session within subject

all_results = []

for region in REGIONS:
    for power_col in POWER_COMPONENTS:
        dv      = power_col   # raw SSL, no z-scoring
        reg_df  = df[df['region'] == region].dropna(subset=[dv]).copy()

        print(f"\n{'='*65}")
        print(f"  Region: {region}  |  DV: {dv} (raw SSL)")
        print(f"  N rows: {len(reg_df):,}  |  N subjects (both phases): {reg_df['subject'].nunique()}")
        print(f"  Sources in this subset: {reg_df['source'].value_counts().to_dict()}")
        print(f"{'='*65}")

        formula = f'{dv} ~ source_code * freq_hz'

        try:
            model = smf.mixedlm(
                formula,
                data      = reg_df,
                groups    = reg_df['subject'],
                vc_formula= {'subj_sess': '0 + C(subj_sess)'}
            )
            result = model.fit(reml=True, method='lbfgs', maxiter=1000)

            print(result.summary())

            # Extract fixed effects
            fe = result.fe_params
            pv = result.pvalues
            ci = result.conf_int()

            for param in fe.index:
                all_results.append({
                    'region'      : region,
                    'power'       : power_col,
                    'parameter'   : param,
                    'beta'        : fe[param],
                    'pval'        : pv[param],
                    'ci_low'      : ci.loc[param, 0],
                    'ci_high'     : ci.loc[param, 1],
                    'n_rows'      : len(reg_df),
                    'n_subjects'  : reg_df['subject'].nunique(),
                    'n_subj_sess' : reg_df['subj_sess'].nunique(),
                    'converged'   : result.converged,
                })

        except Exception as e:
            print(f"  ✗ Model failed: {e}")
            import traceback; traceback.print_exc()
            continue

# ============================================================================
# RESULTS DATAFRAME + FDR CORRECTION
# ============================================================================
results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    print("\n⚠ No results to save.")
else:
    # FDR correction separately for each parameter across region × power combos
    for param in results_df['parameter'].unique():
        mask = results_df['parameter'] == param
        pvals = results_df.loc[mask, 'pval'].values
        if len(pvals) > 1:
            _, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
            results_df.loc[mask, 'pval_fdr'] = pvals_fdr
        else:
            results_df.loc[mask, 'pval_fdr'] = pvals

    results_df['sig_fdr'] = results_df['pval_fdr'] < 0.05
    results_df['sig_raw'] = results_df['pval']      < 0.05

    # Save results
    results_csv = os.path.join(OUTPUT_DIR, 'lme_results_gamma_source_comparison.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"\n✓ Results saved: {results_csv}")
    print(results_df.to_string(index=False))

    # ============================================================================
    # FOREST PLOT  — one panel per region, rows = parameters, colour = power
    # ============================================================================
    params_to_plot = ['source_code', 'freq_hz', 'source_code:freq_hz']
    param_labels   = {
        'source_code'          : 'Source\n(REC_WORD vs RS_Recall)',
        'freq_hz'              : 'Frequency (Hz)',
        'source_code:freq_hz'  : 'Source × Frequency\nInteraction',
    }
    colors = {'frac_ssl': '#4C72B0', 'osc_ssl': '#DD8452'}

    fig, axes = plt.subplots(1, len(REGIONS), figsize=(5 * len(REGIONS), 5), sharey=True)
    if len(REGIONS) == 1:
        axes = [axes]

    for ax, region in zip(axes, REGIONS):
        sub = results_df[results_df['region'] == region].copy()
        sub = sub[sub['parameter'].isin(params_to_plot)]

        y_positions = {}
        spacing = 0.25
        for i, param in enumerate(params_to_plot):
            y_positions[param] = i

        for power_col, offset in zip(POWER_COMPONENTS, [-spacing/2, spacing/2]):
            pwr_sub = sub[sub['power'] == power_col]
            for _, row in pwr_sub.iterrows():
                y   = y_positions[row['parameter']] + offset
                col = colors[power_col]
                marker = '*' if row['sig_fdr'] else ('o' if row['sig_raw'] else 's')
                ax.errorbar(
                    x     = row['beta'],
                    y     = y,
                    xerr  = [[row['beta'] - row['ci_low']],
                              [row['ci_high'] - row['beta']]],
                    fmt   = marker,
                    color = col,
                    markersize = 9 if row['sig_fdr'] else 7,
                    capsize= 4,
                    linewidth= 1.5,
                    label = power_col if row['parameter'] == params_to_plot[0] else ''
                )

        ax.axvline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_yticks(list(y_positions.values()))
        ax.set_yticklabels([param_labels.get(p, p) for p in params_to_plot], fontsize=10)
        ax.set_xlabel('β  (95% CI)  —  raw SSL units', fontsize=11)
        ax.set_title(f'{region}', fontsize=13, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)

    # Legend
    handles = [mpatches.Patch(color=colors[p], label=p) for p in POWER_COMPONENTS]
    handles += [
        plt.Line2D([0],[0], marker='*', color='gray', linestyle='', markersize=9, label='p_FDR < 0.05'),
        plt.Line2D([0],[0], marker='o', color='gray', linestyle='', markersize=7, label='p_raw < 0.05'),
        plt.Line2D([0],[0], marker='s', color='gray', linestyle='', markersize=7, label='n.s.'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, -0.08), frameon=False)

    fig.suptitle(
        'Hippocampal Gamma Power: Spatial Recall (RS_Recall) vs Verbal Recall (REC_WORD)\n'
        f'(correct recalls only, freq > {GAMMA_CUTOFF_HZ} Hz, raw SSL, LME nested random effects)',
        fontsize=12, y=1.02
    )
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'forest_plot_gamma_source_comparison.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Forest plot saved: {plot_path}")

    # ============================================================================
    # DESCRIPTIVE SUMMARY  — mean ± SEM per source × region × freq_hz
    # ============================================================================
    print("\nDescriptive summary (mean ± SEM per source × region):")
    desc = (
        df.groupby(['source', 'region'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
    )
    print(desc.to_string())

    desc_freq = (
        df.groupby(['source', 'region', 'freq_hz'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
        .reset_index()
    )
    desc_csv = os.path.join(OUTPUT_DIR, 'descriptive_gamma_source_by_freq.csv')
    desc_freq.to_csv(desc_csv, index=False)
    print(f"\n✓ Descriptive CSV saved: {desc_csv}")

Loading ./merged_irasa_rs_recall_vs_retrieval.csv ...
  Total rows       : 68,598
  After recalled==1: 65,646
  After freq>40 Hz : 21,882
  Gamma freqs used : [42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

  Subjects with RS_Recall only        : 19
  Subjects with REC_WORD_retrieval only: 2
  Subjects with BOTH (kept)            : 24
  Rows after subject filter            : 17,334

  Sources          : {'RS_Recall': 9048, 'REC_WORD_retrieval': 8286}
  Subjects         : 24
  subj_sess groups : 63

Using raw SSL values (no z-scoring).

  Region: LHP  |  DV: frac_ssl (raw SSL)
  N rows: 7,284  |  N subjects (both phases): 16
  Sources in this subset: {'REC_WORD_retrieval': 4164, 'RS_Recall': 3120}
             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    frac_ssl
No. Observations:    7284       Method:                REML    
No. Groups:          16         Scale:                 0.0460  
Min. group size:     180        Log-Likelihood:        

In [97]:
"""
LME Analysis: Gamma power ~ source * freq_hz * hemisphere

Model (run separately for frac_ssl and osc_ssl):
-------------------------------------------------
    power ~ source_code * freq_hz * hemi_code
            + (1 | subject) + (1 | subject:session)

Fixed effects:
    - source_code  : RS_Recall=0 (spatial recall, reference)
                     REC_WORD_retrieval=1 (verbal recall)
    - freq_hz      : continuous, gamma only (>40 Hz)
    - hemi_code    : LHP=0 (reference), RHP=1

Random effects:
    - subject            : top-level random intercept
    - session within subject : nested random intercept

Filters:
    - recalled == 1 only (correct recalls)
    - freq_hz > 40 Hz
    - subjects with BOTH sources only
    - regions: LHP + RHP pooled into single model with hemisphere factor

Output:
    - CSV of fixed effects (beta, CI, p_raw, p_FDR)
    - Forest plot of all terms
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_CSV   = './merged_irasa_rs_recall_vs_retrieval.csv'
OUTPUT_DIR  = './lme_rs_recall_vs_retrieval_hemi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

POWER_COMPONENTS = ['frac_ssl', 'osc_ssl']
GAMMA_CUTOFF_HZ  = 40.0
REFERENCE_SOURCE = 'RS_Recall'           # source_code = 0
OTHER_SOURCE     = 'REC_WORD_retrieval'  # source_code = 1
REFERENCE_HEMI   = 'LHP'                 # hemi_code   = 0
OTHER_HEMI       = 'RHP'                 # hemi_code   = 1

# ============================================================================
# LOAD & FILTER
# ============================================================================
print(f"Loading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Total rows        : {len(df):,}")

# Correct recalls only
df = df[df['recalled'] == 1].copy()
print(f"  After recalled==1 : {len(df):,}")

# Gamma only
df = df[df['freq_hz'] > GAMMA_CUTOFF_HZ].copy()
print(f"  After freq >40 Hz : {len(df):,}")
print(f"  Gamma freqs used  : {sorted(df['freq_hz'].unique())}")

# LHP + RHP only
df = df[df['region'].isin([REFERENCE_HEMI, OTHER_HEMI])].copy()

# Keep only subjects with BOTH sources
subjects_rs   = set(df[df['source'] == REFERENCE_SOURCE]['subject'].unique())
subjects_ext  = set(df[df['source'] == OTHER_SOURCE]['subject'].unique())
subjects_both = subjects_rs & subjects_ext

print(f"\n  Subjects with RS_Recall only         : {len(subjects_rs  - subjects_ext)}")
print(f"  Subjects with REC_WORD_retrieval only : {len(subjects_ext - subjects_rs)}")
print(f"  Subjects with BOTH (kept)             : {len(subjects_both)}")

df = df[df['subject'].isin(subjects_both)].copy()
print(f"  Rows after subject filter             : {len(df):,}")

# ============================================================================
# DUMMY CODING
# ============================================================================
# source_code : RS_Recall=0 (reference), REC_WORD_retrieval=1
df['source_code'] = (df['source'] == OTHER_SOURCE).astype(int)

# hemi_code : LHP=0 (reference), RHP=1
df['hemi_code'] = (df['region'] == OTHER_HEMI).astype(int)

# Nested grouping for random effects
df['subj_sess'] = df['subject'].astype(str) + '_' + df['session'].astype(str)

print(f"\n  source_code distribution : {df['source_code'].value_counts().to_dict()}")
print(f"  hemi_code distribution   : {df['hemi_code'].value_counts().to_dict()}")
print(f"  Subjects                 : {df['subject'].nunique()}")
print(f"  subj_sess groups         : {df['subj_sess'].nunique()}")

# ============================================================================
# LME MODELS  — one per power component
# ============================================================================
# Full factorial model:
#   power ~ source_code * freq_hz * hemi_code
#           + (1|subject) + (1|subject:session)
#
# This gives 8 fixed-effect terms:
#   Intercept                          → LHP, RS_Recall, at freq_hz=0
#   source_code                        → verbal vs spatial, in LHP
#   freq_hz                            → gamma slope, in LHP, RS_Recall
#   hemi_code                          → RHP vs LHP, in RS_Recall
#   source_code:freq_hz                → does freq slope differ by source, in LHP
#   source_code:hemi_code              → source effect differs LHP vs RHP
#   freq_hz:hemi_code                  → freq slope differs LHP vs RHP
#   source_code:freq_hz:hemi_code      → 3-way: source×freq interaction differs by hemisphere

all_results = []

for power_col in POWER_COMPONENTS:
    reg_df = df.dropna(subset=[power_col]).copy()

    print(f"\n{'='*70}")
    print(f"  DV: {power_col} (raw SSL)  |  N rows: {len(reg_df):,}")
    print(f"  N subjects: {reg_df['subject'].nunique()}  |  "
          f"N subj_sess: {reg_df['subj_sess'].nunique()}")
    print(f"  Rows per source × hemi:")
    print(reg_df.groupby(['source', 'region']).size().to_string())
    print(f"{'='*70}")

    formula = (f'{power_col} ~ source_code * freq_hz * hemi_code')

    try:
        model = smf.mixedlm(
            formula,
            data       = reg_df,
            groups     = reg_df['subject'],
            vc_formula = {'subj_sess': '0 + C(subj_sess)'}
        )
        result = model.fit(reml=True, method='lbfgs', maxiter=1000)

        print(result.summary())

        fe = result.fe_params
        pv = result.pvalues
        ci = result.conf_int()

        for param in fe.index:
            all_results.append({
                'power'       : power_col,
                'parameter'   : param,
                'beta'        : fe[param],
                'pval'        : pv[param],
                'ci_low'      : ci.loc[param, 0],
                'ci_high'     : ci.loc[param, 1],
                'n_rows'      : len(reg_df),
                'n_subjects'  : reg_df['subject'].nunique(),
                'n_subj_sess' : reg_df['subj_sess'].nunique(),
                'converged'   : result.converged,
            })

    except Exception as e:
        print(f"  ✗ Model failed: {e}")
        import traceback; traceback.print_exc()
        continue

# ============================================================================
# RESULTS + FDR CORRECTION
# ============================================================================
results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    print("\n⚠ No results to save.")
else:
    # FDR across power components per parameter
    for param in results_df['parameter'].unique():
        mask  = results_df['parameter'] == param
        pvals = results_df.loc[mask, 'pval'].values
        if len(pvals) > 1:
            _, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
            results_df.loc[mask, 'pval_fdr'] = pvals_fdr
        else:
            results_df.loc[mask, 'pval_fdr'] = pvals

    results_df['sig_fdr'] = results_df['pval_fdr'] < 0.05
    results_df['sig_raw'] = results_df['pval']      < 0.05

    results_csv = os.path.join(OUTPUT_DIR, 'lme_results_gamma_hemi_source.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"\n✓ Results saved: {results_csv}")
    print(results_df.to_string(index=False))

    # ============================================================================
    # FOREST PLOT
    # ============================================================================
    # Show all terms except Intercept
    params_to_plot = [
        'source_code',
        'freq_hz',
        'hemi_code',
        'source_code:freq_hz',
        'source_code:hemi_code',
        'freq_hz:hemi_code',
        'source_code:freq_hz:hemi_code',
    ]
    param_labels = {
        'source_code'                  : 'Source\n(verbal vs spatial)',
        'freq_hz'                      : 'Frequency (Hz)',
        'hemi_code'                    : 'Hemisphere\n(RHP vs LHP)',
        'source_code:freq_hz'          : 'Source × Freq',
        'source_code:hemi_code'        : 'Source × Hemisphere',
        'freq_hz:hemi_code'            : 'Freq × Hemisphere',
        'source_code:freq_hz:hemi_code': 'Source × Freq\n× Hemisphere',
    }
    colors = {'frac_ssl': '#4C72B0', 'osc_ssl': '#DD8452'}

    fig, ax = plt.subplots(figsize=(7, 7))
    spacing = 0.25

    for i, param in enumerate(params_to_plot):
        for j, power_col in enumerate(POWER_COMPONENTS):
            offset  = (j - 0.5) * spacing
            sub_row = results_df[
                (results_df['parameter'] == param) &
                (results_df['power']     == power_col)
            ]
            if len(sub_row) == 0:
                continue
            row = sub_row.iloc[0]
            y   = i + offset
            col = colors[power_col]
            marker = '*' if row['sig_fdr'] else ('o' if row['sig_raw'] else 's')

            ax.errorbar(
                x          = row['beta'],
                y          = y,
                xerr       = [[row['beta'] - row['ci_low']],
                               [row['ci_high'] - row['beta']]],
                fmt        = marker,
                color      = col,
                markersize = 10 if row['sig_fdr'] else 7,
                capsize    = 4,
                linewidth  = 1.5,
                label      = power_col if i == 0 else ''
            )

    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(params_to_plot)))
    ax.set_yticklabels(
        [param_labels.get(p, p) for p in params_to_plot], fontsize=10
    )
    ax.set_xlabel('β  (95% CI)  —  raw SSL units', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)

    handles = [mpatches.Patch(color=colors[p], label=p) for p in POWER_COMPONENTS]
    handles += [
        plt.Line2D([0],[0], marker='*', color='gray', linestyle='', markersize=10, label='p_FDR < 0.05'),
        plt.Line2D([0],[0], marker='o', color='gray', linestyle='', markersize=7,  label='p_raw < 0.05'),
        plt.Line2D([0],[0], marker='s', color='gray', linestyle='', markersize=7,  label='n.s.'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, -0.06), frameon=False)

    ax.set_title(
        'Hippocampal Gamma Power: Source × Hemisphere\n'
        f'(correct recalls, freq > {GAMMA_CUTOFF_HZ} Hz, raw SSL, '
        f'LHP=reference, RS_Recall=reference)',
        fontsize=11
    )
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'forest_plot_gamma_hemi_source.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Forest plot saved: {plot_path}")

    # ============================================================================
    # DESCRIPTIVE SUMMARY
    # ============================================================================
    desc_freq = (
        df.groupby(['source', 'region', 'freq_hz'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
        .reset_index()
    )
    desc_csv = os.path.join(OUTPUT_DIR, 'descriptive_gamma_hemi_source_by_freq.csv')
    desc_freq.to_csv(desc_csv, index=False)
    print(f"✓ Descriptive CSV saved: {desc_csv}")

Loading ./merged_irasa_rs_recall_vs_retrieval.csv ...
  Total rows        : 68,598
  After recalled==1 : 65,646
  After freq >40 Hz : 21,882
  Gamma freqs used  : [42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

  Subjects with RS_Recall only         : 19
  Subjects with REC_WORD_retrieval only : 2
  Subjects with BOTH (kept)             : 24
  Rows after subject filter             : 17,334

  source_code distribution : {0: 9048, 1: 8286}
  hemi_code distribution   : {0: 8688, 1: 8646}
  Subjects                 : 24
  subj_sess groups         : 63

  DV: frac_ssl (raw SSL)  |  N rows: 14,880
  N subjects: 24  |  N subj_sess: 63
  Rows per source × hemi:
source              region
REC_WORD_retrieval  LHP       4164
                    RHP       4122
RS_Recall           LHP       3120
                    RHP       3474
                  Mixed Linear Model Regression Results
Model:                   MixedLM      Dependent Variable:      frac_ssl  
No. Observations:        14880        Method:

In [99]:
import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from scipy.spatial.distance import euclidean
import copy

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly']

output_dir = './subject_results_clustering_sr_rec_word'
os.makedirs(output_dir, exist_ok=True)

# ============================================================================
# CLUSTERING HELPER FUNCTIONS
# ============================================================================

def compute_euclidean_distance(coord1, coord2):
    return euclidean(coord1, coord2)

def build_spatial_distance_matrix(presented_coordinates):
    n_items = len(presented_coordinates)
    distance_matrix = np.zeros((n_items, n_items))
    for i in range(n_items):
        for j in range(n_items):
            distance_matrix[i, j] = compute_euclidean_distance(
                presented_coordinates[i], presented_coordinates[j]
            )
    return distance_matrix

def build_temporal_distance_matrix(n_items):
    temporal_distance_matrix = np.zeros((n_items, n_items))
    for i in range(n_items):
        for j in range(n_items):
            temporal_distance_matrix[i, j] = abs(i - j)
    return temporal_distance_matrix

def convert_recalled_coords_to_itemnos(presented_coordinates, recalled_coordinates, tolerance=0.01):
    recalled_itemnos = []
    for recalled_coord in recalled_coordinates:
        distances = np.array([
            compute_euclidean_distance(recalled_coord, pres_coord)
            for pres_coord in presented_coordinates
        ])
        closest_idx = np.argmin(distances)
        if distances[closest_idx] <= tolerance:
            recalled_itemnos.append(closest_idx + 1)
        else:
            recalled_itemnos.append(-1)
    return np.array(recalled_itemnos)

def make_recalls_matrix(pres_itemnos, rec_itemnos):
    n_trials = pres_itemnos.shape[0]
    n_items = pres_itemnos.shape[1]
    n_recalls = rec_itemnos.shape[1]

    recalls = np.zeros([n_trials, n_recalls], dtype=int)

    for trial in range(n_trials):
        for recall in range(n_recalls):
            val = rec_itemnos[trial, recall]
            if val == 0 or np.isnan(val):
                continue
            elif val > 0:
                serialpos = np.where(val == pres_itemnos[trial, :])[0] + 1
                if len(serialpos) > 1:
                    raise Exception('An item was presented more than once.')
                elif len(serialpos) == 0:
                    recalls[trial, recall] = -1
                else:
                    recalls[trial, recall] = serialpos[0]
            else:
                recalls[trial, recall] = -1
    return recalls

def make_clean_recalls_mask2d(data):
    result = copy.deepcopy(data)
    for num, item in enumerate(data):
        seen = []
        for index, recall in enumerate(item):
            if recall > 0 and recall not in seen:
                result[num][index] = 1
                seen.append(recall)
            else:
                result[num][index] = 0
    return result

def dist_percentile_rank(actual, possible):
    if len(possible) < 2:
        return None
    possible_sorted = sorted(possible)[::-1]
    matches = np.where(np.array(possible_sorted) == actual)[0]
    if len(matches) > 0:
        rank = np.mean(matches)
        ptile_rank = rank / (len(possible_sorted) - 1.)
    else:
        ptile_rank = None
    return ptile_rank

def dist_fact_single_trial(rec_itemnos, pres_itemnos, dist_mat, skip_first_n=0):
    rec_itemnos = np.array(rec_itemnos, dtype=float)
    pres_itemnos = np.array(pres_itemnos, dtype=int)
    dist_mat = np.array(dist_mat)

    if rec_itemnos.ndim == 2:
        rec_itemnos = rec_itemnos[0]
    if pres_itemnos.ndim == 2:
        pres_itemnos = pres_itemnos[0]

    rec_itemnos_2d = rec_itemnos.reshape(1, -1)
    pres_itemnos_2d = pres_itemnos.reshape(1, -1)

    clean_recalls_mask = make_clean_recalls_mask2d(
        make_recalls_matrix(pres_itemnos_2d, rec_itemnos_2d)
    )[0]

    transition_scores = []
    seen = set()

    for j, rec in enumerate(rec_itemnos[:-1]):
        rec = int(rec)
        seen.add(rec)

        if clean_recalls_mask[j] and clean_recalls_mask[j + 1] and j >= skip_first_n:
            possibles = np.array([
                dist_mat[rec - 1, int(poss_rec) - 1]
                for poss_rec in pres_itemnos if int(poss_rec) not in seen
            ])
            next_rec = int(rec_itemnos[j + 1])
            actual = dist_mat[rec - 1, next_rec - 1]
            ptile_rank = dist_percentile_rank(actual, possibles)
            if ptile_rank is not None:
                transition_scores.append(ptile_rank)

    return np.array(transition_scores)

def compute_clustering_scores(presented_coordinates, recalled_coordinates, rec_itemnos_array):
    """
    Returns (spatial_scores, temporal_scores), each an array of length
    equal to the number of valid transitions (len(clean_rec_itemnos) - 1).
    """
    spatial_dist_mat = build_spatial_distance_matrix(presented_coordinates)
    n_items = len(presented_coordinates)
    temporal_dist_mat = build_temporal_distance_matrix(n_items)

    pres_itemnos = np.array([list(range(1, n_items + 1))])
    rec_itemnos = np.zeros((1, n_items), dtype=int)
    rec_itemnos[0, :len(rec_itemnos_array)] = rec_itemnos_array

    spatial_scores = dist_fact_single_trial(rec_itemnos, pres_itemnos, spatial_dist_mat)
    temporal_scores = dist_fact_single_trial(rec_itemnos, pres_itemnos, temporal_dist_mat)

    return spatial_scores, temporal_scores

# ============================================================================
# MAIN ANALYSIS
# ============================================================================

whole_df = cml.CMLReader.get_data_index()

all_rows = []

for exp in EXPERIMENTS:
    subjects = whole_df.query('experiment == @exp')['subject'].unique()
    print(f"\n{'='*80}")
    print(f"Experiment: {exp}  |  Subjects: {len(subjects)}")
    print(f"{'='*80}")

    for subj_idx, subject in enumerate(subjects):
        print(f"\n[{subj_idx+1}/{len(subjects)}] Subject: {subject}")

        sub_df = whole_df.query('experiment == @exp and subject == @subject')
        sessions = sub_df['session'].unique()

        for session in sessions:
            print(f"  Session: {session}")

            try:
                reader = cml.CMLReader(subject, exp, session=session)
                evs = reader.load('task_events')

                sr_rec_evs = evs[evs['type'] == 'SR_REC_WORD'].copy()
                word_evs = evs[evs['type'] == 'WORD'].copy()

                if len(sr_rec_evs) == 0:
                    print(f"    ⚠ No SR_REC_WORD events found, skipping.")
                    continue

                trials = sr_rec_evs['trial'].unique()

                for trial in trials:
                    if trial == -999 or trial < 0:
                        continue

                    trial_sr_rec = sr_rec_evs[sr_rec_evs['trial'] == trial]
                    trial_word   = word_evs[word_evs['trial'] == trial]

                    if len(trial_sr_rec) == 0 or len(trial_word) == 0:
                        continue

                    # Build coordinate arrays
                    recalled_coordinates = np.column_stack([
                        trial_sr_rec['storeX'].values,
                        trial_sr_rec['storeZ'].values
                    ])
                    presented_coordinates = np.column_stack([
                        trial_word['storeX'].values,
                        trial_word['storeZ'].values
                    ])

                    # Map recalled coordinates → item numbers
                    rec_itemnos_array = convert_recalled_coords_to_itemnos(
                        presented_coordinates, recalled_coordinates
                    )

                    # Build clean recalls mask
                    n_items = len(presented_coordinates)
                    pres_itemnos_2d = np.array([list(range(1, n_items + 1))])
                    rec_itemnos_2d = np.zeros((1, n_items), dtype=int)
                    rec_itemnos_2d[0, :len(rec_itemnos_array)] = rec_itemnos_array

                    clean_mask = make_clean_recalls_mask2d(
                        make_recalls_matrix(pres_itemnos_2d, rec_itemnos_2d)
                    )[0]
                    clean_recall_indices = np.where(clean_mask == 1)[0]
                    clean_rec_itemnos = rec_itemnos_array[clean_recall_indices]

                    if len(clean_rec_itemnos) == 0:
                        print(f"    ⚠ Trial {trial}: no clean recalls, skipping.")
                        continue

                    # Compute clustering scores (one per transition, len = N_clean - 1)
                    spatial_scores, temporal_scores = compute_clustering_scores(
                        presented_coordinates,
                        recalled_coordinates[clean_recall_indices],
                        clean_rec_itemnos
                    )

                    # Assign scores to each clean recalled word
                    clean_sr_rec = trial_sr_rec.iloc[clean_recall_indices].reset_index(drop=True)

                    for clean_idx, rec_itemno in enumerate(clean_rec_itemnos):
                        row = clean_sr_rec.iloc[clean_idx]

                        sp_from_prev = spatial_scores[clean_idx - 1]  if clean_idx > 0 and clean_idx - 1 < len(spatial_scores)  else np.nan
                        sp_to_next   = spatial_scores[clean_idx]      if clean_idx < len(spatial_scores)                         else np.nan
                        t_from_prev  = temporal_scores[clean_idx - 1] if clean_idx > 0 and clean_idx - 1 < len(temporal_scores)  else np.nan
                        t_to_next    = temporal_scores[clean_idx]      if clean_idx < len(temporal_scores)                        else np.nan

                        all_rows.append({
                            'experiment':            exp,
                            'subject':               subject,
                            'session':               session,
                            'trial':                 trial,
                            'clean_idx':             clean_idx,
                            'eegoffset':             row.get('eegoffset', np.nan),
                            'item':                  row.get('item', ''),
                            'storeX':                row.get('storeX', np.nan),
                            'storeZ':                row.get('storeZ', np.nan),
                            'itemno':                rec_itemno,
                            'recalled':              1 if rec_itemno > 0 else 0,
                            'SP_clustering_from_prev': sp_from_prev,
                            'SP_clustering_to_next':   sp_to_next,
                            'T_clustering_from_prev':  t_from_prev,
                            'T_clustering_to_next':    t_to_next,
                        })

                    print(f"      ✓ Trial {trial}: {len(clean_rec_itemnos)} clean recalls")

            except Exception as e:
                import traceback
                print(f"  ✗ Session {session} FAILED: {e}")
                traceback.print_exc()
                continue

# ============================================================================
# SAVE MASTER CSV
# ============================================================================

if len(all_rows) > 0:
    master_df = pd.DataFrame(all_rows)
    master_csv = os.path.join(output_dir, 'ALL_SUBJECTS_clustering_sr_rec_word.csv')
    master_df.to_csv(master_csv, index=False)

    print(f"\n{'='*80}")
    print(f"✓ Saved: {master_csv}")
    print(f"  Rows:        {len(master_df):,}")
    print(f"  Subjects:    {master_df['subject'].nunique()}")
    print(f"  Experiments: {master_df['experiment'].unique()}")
    print(master_df.head(10))
else:
    print("\n⚠ No data collected.")


Experiment: DBOY1  |  Subjects: 44

[1/44] Subject: R1494D
  Session: 0
  Session: 1
  Session: 2

[2/44] Subject: R1501J
  Session: 0
  Session: 1

[3/44] Subject: R1502D
  Session: 0
  Session: 1

[4/44] Subject: R1503E
  Session: 0
  Session: 1
  Session: 2

[5/44] Subject: R1504E
  Session: 0

[6/44] Subject: R1509E
  Session: 0
    ⚠ No SR_REC_WORD events found, skipping.
  Session: 1
  Session: 2
  Session: 3
    ⚠ No SR_REC_WORD events found, skipping.
  Session: 4
  Session: 5

[7/44] Subject: R1513D
  Session: 0
    ⚠ No SR_REC_WORD events found, skipping.

[8/44] Subject: R1520T
  Session: 0

[9/44] Subject: R1521E
  Session: 0
  Session: 1
  Session: 2

[10/44] Subject: R1523E
  Session: 0
  Session: 1

[11/44] Subject: R1524T
  Session: 0
  Session: 1
  Session: 2

[12/44] Subject: R1526J
  Session: 3
    ⚠ No SR_REC_WORD events found, skipping.
  Session: 5
  Session: 6
  Session: 7

[13/44] Subject: R1529T
  Session: 1
  Session: 2

[14/44] Subject: R1530J
  Session: 0
 

In [101]:
if len(all_rows) > 0:
    master_df = pd.DataFrame(all_rows)
    master_csv = os.path.join(output_dir, 'ALL_SUBJECTS_clustering_sr_rec_word.csv')
    master_df.to_csv(master_csv, index=False)

    print(f"\n{'='*80}")
    print(f"✓ Saved: {master_csv}")
    print(f"  Rows:        {len(master_df):,}")
    print(f"  Subjects:    {master_df['subject'].nunique()}")
    print(f"  Experiments: {master_df['experiment'].unique()}")
    print(master_df.head(10))
else:
    print("\n⚠ No data collected.")


⚠ No data collected.


In [103]:
import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from scipy.spatial.distance import euclidean
import copy

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly']

output_dir = './subject_results_clustering_sr_rec_word'
os.makedirs(output_dir, exist_ok=True)

# ============================================================================
# CLUSTERING HELPER FUNCTIONS
# ============================================================================

def compute_euclidean_distance(coord1, coord2):
    return euclidean(coord1, coord2)

def build_spatial_distance_matrix(presented_coordinates):
    n_items = len(presented_coordinates)
    distance_matrix = np.zeros((n_items, n_items))
    for i in range(n_items):
        for j in range(n_items):
            distance_matrix[i, j] = compute_euclidean_distance(
                presented_coordinates[i], presented_coordinates[j]
            )
    return distance_matrix

def build_temporal_distance_matrix(n_items):
    temporal_distance_matrix = np.zeros((n_items, n_items))
    for i in range(n_items):
        for j in range(n_items):
            temporal_distance_matrix[i, j] = abs(i - j)
    return temporal_distance_matrix

def convert_recalled_coords_to_itemnos(presented_coordinates, recalled_coordinates, tolerance=0.01):
    recalled_itemnos = []
    for recalled_coord in recalled_coordinates:
        distances = np.array([
            compute_euclidean_distance(recalled_coord, pres_coord)
            for pres_coord in presented_coordinates
        ])
        closest_idx = np.argmin(distances)
        if distances[closest_idx] <= tolerance:
            recalled_itemnos.append(closest_idx + 1)
        else:
            recalled_itemnos.append(-1)
    return np.array(recalled_itemnos)

def make_recalls_matrix(pres_itemnos, rec_itemnos):
    n_trials = pres_itemnos.shape[0]
    n_items = pres_itemnos.shape[1]
    n_recalls = rec_itemnos.shape[1]

    recalls = np.zeros([n_trials, n_recalls], dtype=int)

    for trial in range(n_trials):
        for recall in range(n_recalls):
            val = rec_itemnos[trial, recall]
            if val == 0 or np.isnan(val):
                continue
            elif val > 0:
                serialpos = np.where(val == pres_itemnos[trial, :])[0] + 1
                if len(serialpos) > 1:
                    raise Exception('An item was presented more than once.')
                elif len(serialpos) == 0:
                    recalls[trial, recall] = -1
                else:
                    recalls[trial, recall] = serialpos[0]
            else:
                recalls[trial, recall] = -1
    return recalls

def make_clean_recalls_mask2d(data):
    result = copy.deepcopy(data)
    for num, item in enumerate(data):
        seen = []
        for index, recall in enumerate(item):
            if recall > 0 and recall not in seen:
                result[num][index] = 1
                seen.append(recall)
            else:
                result[num][index] = 0
    return result

def dist_percentile_rank(actual, possible):
    if len(possible) < 2:
        return None
    possible_sorted = sorted(possible)[::-1]
    matches = np.where(np.array(possible_sorted) == actual)[0]
    if len(matches) > 0:
        rank = np.mean(matches)
        ptile_rank = rank / (len(possible_sorted) - 1.)
    else:
        ptile_rank = None
    return ptile_rank

def dist_fact_single_trial(rec_itemnos, pres_itemnos, dist_mat, skip_first_n=0):
    rec_itemnos = np.array(rec_itemnos, dtype=float)
    pres_itemnos = np.array(pres_itemnos, dtype=int)
    dist_mat = np.array(dist_mat)

    if rec_itemnos.ndim == 2:
        rec_itemnos = rec_itemnos[0]
    if pres_itemnos.ndim == 2:
        pres_itemnos = pres_itemnos[0]

    rec_itemnos_2d = rec_itemnos.reshape(1, -1)
    pres_itemnos_2d = pres_itemnos.reshape(1, -1)

    clean_recalls_mask = make_clean_recalls_mask2d(
        make_recalls_matrix(pres_itemnos_2d, rec_itemnos_2d)
    )[0]

    transition_scores = []
    seen = set()

    for j, rec in enumerate(rec_itemnos[:-1]):
        rec = int(rec)
        seen.add(rec)

        if clean_recalls_mask[j] and clean_recalls_mask[j + 1] and j >= skip_first_n:
            possibles = np.array([
                dist_mat[rec - 1, int(poss_rec) - 1]
                for poss_rec in pres_itemnos if int(poss_rec) not in seen
            ])
            next_rec = int(rec_itemnos[j + 1])
            actual = dist_mat[rec - 1, next_rec - 1]
            ptile_rank = dist_percentile_rank(actual, possibles)
            if ptile_rank is not None:
                transition_scores.append(ptile_rank)

    return np.array(transition_scores)

def compute_clustering_scores(presented_coordinates, recalled_coordinates, rec_itemnos_array):
    """
    Returns (spatial_scores, temporal_scores), each an array of length
    equal to the number of valid transitions (len(clean_rec_itemnos) - 1).
    """
    spatial_dist_mat = build_spatial_distance_matrix(presented_coordinates)
    n_items = len(presented_coordinates)
    temporal_dist_mat = build_temporal_distance_matrix(n_items)

    pres_itemnos = np.array([list(range(1, n_items + 1))])
    rec_itemnos = np.zeros((1, n_items), dtype=int)
    rec_itemnos[0, :len(rec_itemnos_array)] = rec_itemnos_array

    spatial_scores = dist_fact_single_trial(rec_itemnos, pres_itemnos, spatial_dist_mat)
    temporal_scores = dist_fact_single_trial(rec_itemnos, pres_itemnos, temporal_dist_mat)

    return spatial_scores, temporal_scores

# ============================================================================
# MAIN ANALYSIS
# ============================================================================

whole_df = cml.CMLReader.get_data_index()

all_rows = []

for exp in EXPERIMENTS:
    subjects = whole_df.query('experiment == @exp')['subject'].unique()
    print(f"\n{'='*80}")
    print(f"Experiment: {exp}  |  Subjects: {len(subjects)}")
    print(f"{'='*80}")

    for subj_idx, subject in enumerate(subjects):
        print(f"\n[{subj_idx+1}/{len(subjects)}] Subject: {subject}")

        sub_df = whole_df.query('experiment == @exp and subject == @subject')
        sessions = sub_df['session'].unique()

        for session in sessions:
            print(f"  Session: {session}")

            try:
                reader = cml.CMLReader(subject, exp, session=session)
                evs = reader.load('task_events')

                sr_rec_evs = evs[evs['type'] == 'SR_REC_WORD'].copy()
                word_evs = evs[evs['type'] == 'WORD'].copy()

                if len(sr_rec_evs) == 0:
                    print(f"    ⚠ No SR_REC_WORD events found, skipping.")
                    continue

                if len(word_evs) == 0:
                    print(f"    ⚠ No WORD events found, skipping.")
                    continue

                # ----------------------------------------------------------------
                # SR_REC_WORD.trial is always -999; SR_REC_WORD.itemno is the
                # 1-based serialpos into the full presented word list.
                # WORD events have real trial numbers and storeX/Z coordinates.
                # Strategy:
                #   1. Build a serialpos → (storeX, storeZ) lookup from WORD events.
                #   2. Treat the entire session as one recall sequence (SR is
                #      session-level free recall across all trials).
                #   3. Use itemno directly as the serial position; no coordinate
                #      matching needed.
                # ----------------------------------------------------------------

                # Build serialpos → store coords lookup from WORD events
                # serialpos in WORD is 1-based within each trial; we need a
                # session-wide unique index. Use (trial, serialpos) → global itemno.
                # But SR_REC_WORD.itemno is already a session-wide item index,
                # so we build: session_itemno → (storeX, storeZ).
                # WORD events are ordered; assign a global 1-based index by order.
                word_evs_sorted = word_evs.sort_values('eegoffset').reset_index(drop=True)
                n_items = len(word_evs_sorted)

                # Map: global_itemno (1-based) → store coordinates
                itemno_to_coords = {}
                for i, wrow in word_evs_sorted.iterrows():
                    global_itemno = i + 1  # 1-based
                    itemno_to_coords[global_itemno] = (wrow['storeX'], wrow['storeZ'])

                # Build presented_coordinates array (ordered by global itemno)
                presented_coordinates = np.array([
                    itemno_to_coords[k] for k in sorted(itemno_to_coords.keys())
                ])

                # SR_REC_WORD.itemno is the global item index (1-based)
                # Filter out intrusions (itemno <= 0 or > n_items)
                sr_valid = sr_rec_evs[
                    (sr_rec_evs['itemno'] > 0) & (sr_rec_evs['itemno'] <= n_items)
                ].copy().reset_index(drop=True)

                if len(sr_valid) == 0:
                    print(f"    ⚠ No valid SR_REC_WORD itemnos, skipping.")
                    continue

                rec_itemnos_array = sr_valid['itemno'].values  # already 1-based

                # Build clean recalls mask (removes repetitions)
                pres_itemnos_2d = np.array([list(range(1, n_items + 1))])
                rec_itemnos_2d  = np.zeros((1, n_items), dtype=int)
                rec_itemnos_2d[0, :len(rec_itemnos_array)] = rec_itemnos_array

                clean_mask = make_clean_recalls_mask2d(
                    make_recalls_matrix(pres_itemnos_2d, rec_itemnos_2d)
                )[0]
                clean_recall_indices = np.where(clean_mask == 1)[0]
                clean_rec_itemnos    = rec_itemnos_array[clean_recall_indices]

                if len(clean_rec_itemnos) == 0:
                    print(f"    ⚠ No clean recalls, skipping.")
                    continue

                # Compute clustering (spatial uses store coords, temporal uses serialpos)
                spatial_scores, temporal_scores = compute_clustering_scores(
                    presented_coordinates,
                    presented_coordinates[clean_rec_itemnos - 1],  # coords in recall order
                    clean_rec_itemnos
                )

                clean_sr = sr_valid.iloc[clean_recall_indices].reset_index(drop=True)

                for clean_idx, rec_itemno in enumerate(clean_rec_itemnos):
                    row = clean_sr.iloc[clean_idx]
                    store_coords = itemno_to_coords.get(rec_itemno, (np.nan, np.nan))

                    sp_from_prev = spatial_scores[clean_idx - 1]  if clean_idx > 0 and clean_idx - 1 < len(spatial_scores)  else np.nan
                    sp_to_next   = spatial_scores[clean_idx]      if clean_idx     < len(spatial_scores)                    else np.nan
                    t_from_prev  = temporal_scores[clean_idx - 1] if clean_idx > 0 and clean_idx - 1 < len(temporal_scores) else np.nan
                    t_to_next    = temporal_scores[clean_idx]      if clean_idx     < len(temporal_scores)                   else np.nan

                    all_rows.append({
                        'experiment':              exp,
                        'subject':                 subject,
                        'session':                 session,
                        'clean_idx':               clean_idx,
                        'eegoffset':               row['eegoffset'],
                        'item':                    row['item'],
                        'storeX':                  store_coords[0],
                        'storeZ':                  store_coords[1],
                        'itemno':                  rec_itemno,
                        'recalled':                1 if row.get('recalled', -999) != -999 else 0,
                        'SP_clustering_from_prev': sp_from_prev,
                        'SP_clustering_to_next':   sp_to_next,
                        'T_clustering_from_prev':  t_from_prev,
                        'T_clustering_to_next':    t_to_next,
                    })

                print(f"    ✓ {len(clean_rec_itemnos)} clean recalls  |  SP scores: {(~np.isnan(spatial_scores)).sum()}  TC scores: {(~np.isnan(temporal_scores)).sum()}")

            except Exception as e:
                import traceback
                print(f"  ✗ Session {session} FAILED: {e}")
                traceback.print_exc()
                continue

# ============================================================================
# SAVE MASTER CSV
# ============================================================================

if len(all_rows) > 0:
    master_df = pd.DataFrame(all_rows)
    master_csv = os.path.join(output_dir, 'ALL_SUBJECTS_clustering_sr_rec_word.csv')
    master_df.to_csv(master_csv, index=False)

    print(f"\n{'='*80}")
    print(f"✓ Saved: {master_csv}")
    print(f"  Rows:        {len(master_df):,}")
    print(f"  Subjects:    {master_df['subject'].nunique()}")
    print(f"  Experiments: {master_df['experiment'].unique()}")
    print(master_df.head(10))
else:
    print("\n⚠ No data collected.")


Experiment: DBOY1  |  Subjects: 44

[1/44] Subject: R1494D
  Session: 0
    ✓ 6 clean recalls  |  SP scores: 5  TC scores: 5
  Session: 1
    ⚠ No valid SR_REC_WORD itemnos, skipping.
  Session: 2
    ✓ 8 clean recalls  |  SP scores: 7  TC scores: 7

[2/44] Subject: R1501J
  Session: 0
    ✓ 14 clean recalls  |  SP scores: 13  TC scores: 13
  Session: 1
    ✓ 14 clean recalls  |  SP scores: 13  TC scores: 13

[3/44] Subject: R1502D
  Session: 0
    ✓ 11 clean recalls  |  SP scores: 10  TC scores: 10
  Session: 1
    ✓ 8 clean recalls  |  SP scores: 7  TC scores: 7

[4/44] Subject: R1503E
  Session: 0
    ⚠ No valid SR_REC_WORD itemnos, skipping.
  Session: 1
    ✓ 10 clean recalls  |  SP scores: 9  TC scores: 9
  Session: 2
    ✓ 11 clean recalls  |  SP scores: 10  TC scores: 10

[5/44] Subject: R1504E
  Session: 0
    ✓ 11 clean recalls  |  SP scores: 10  TC scores: 10

[6/44] Subject: R1509E
  Session: 0
    ⚠ No SR_REC_WORD events found, skipping.
  Session: 1
    ✓ 14 clean recall

In [104]:
import pandas as pd
import numpy as np
import os

# ============================================================================
# CONFIG
# ============================================================================

input_csv  = './subject_results_clustering_sr_rec_word/ALL_SUBJECTS_clustering_sr_rec_word.csv'
output_dir = './subject_results_clustering_sr_rec_word'

# ============================================================================
# LOAD
# ============================================================================

df = pd.read_csv(input_csv)
print(f"Loaded {len(df):,} rows")
print(f"Subjects: {df['subject'].nunique()}")
print(f"Experiments: {df['experiment'].unique()}")
print(f"Columns: {df.columns.tolist()}\n")

CLUST_COLS = [
    'SP_clustering_from_prev',
    'SP_clustering_to_next',
    'T_clustering_from_prev',
    'T_clustering_to_next',
]

# ============================================================================
# STEP 1: Average within each subject × session
# ============================================================================

session_avg = (
    df
    .groupby(['experiment', 'subject', 'session'])[CLUST_COLS]
    .mean()
    .reset_index()
)
session_avg.columns.name = None

print("=== Session-level averages (first 10 rows) ===")
print(session_avg.head(10).to_string(index=False))
print(f"\nTotal subject×session rows: {len(session_avg)}\n")

# Save session-level averages
session_csv = os.path.join(output_dir, 'session_avg_clustering_sr_rec_word.csv')
session_avg.to_csv(session_csv, index=False)
print(f"✓ Saved session averages: {session_csv}")

# ============================================================================
# STEP 2: Average across sessions within each subject
# ============================================================================

subject_avg = (
    session_avg
    .groupby(['experiment', 'subject'])[CLUST_COLS]
    .mean()
    .reset_index()
)
subject_avg.columns.name = None

# Add number of sessions per subject
n_sessions = session_avg.groupby(['experiment', 'subject'])['session'].count().reset_index()
n_sessions.columns = ['experiment', 'subject', 'n_sessions']
subject_avg = subject_avg.merge(n_sessions, on=['experiment', 'subject'])

print("\n=== Subject-level averages ===")
print(subject_avg.to_string(index=False))

# Save subject-level averages
subject_csv = os.path.join(output_dir, 'subject_avg_clustering_sr_rec_word.csv')
subject_avg.to_csv(subject_csv, index=False)
print(f"\n✓ Saved subject averages: {subject_csv}")

# ============================================================================
# STEP 3: Grand averages (mean ± SEM across subjects) per experiment
# ============================================================================

print("\n=== Grand averages (mean ± SEM across subjects) ===")
for exp, grp in subject_avg.groupby('experiment'):
    print(f"\n  {exp}  (N={len(grp)} subjects)")
    for col in CLUST_COLS:
        m   = grp[col].mean()
        sem = grp[col].sem()
        print(f"    {col:<30s}  {m:.4f} ± {sem:.4f}")

Loaded 845 rows
Subjects: 43
Experiments: ['DBOY1' 'EFRCourierReadOnly']
Columns: ['experiment', 'subject', 'session', 'clean_idx', 'eegoffset', 'item', 'storeX', 'storeZ', 'itemno', 'recalled', 'SP_clustering_from_prev', 'SP_clustering_to_next', 'T_clustering_from_prev', 'T_clustering_to_next']

=== Session-level averages (first 10 rows) ===
experiment subject  session  SP_clustering_from_prev  SP_clustering_to_next  T_clustering_from_prev  T_clustering_to_next
     DBOY1  R1494D        0                 0.533132               0.533132                0.901127              0.901127
     DBOY1  R1494D        2                 0.624756               0.624756                0.872340              0.872340
     DBOY1  R1501J        0                 0.483692               0.483692                0.909179              0.909179
     DBOY1  R1501J        1                 0.460124               0.460124                0.893419              0.893419
     DBOY1  R1502D        0                 0

In [105]:
subject_avg

,experiment,subject,SP_clustering_from_prev,SP_clustering_to_next,T_clustering_from_prev,T_clustering_to_next,n_sessions
0,DBOY1,R1494D,0.578944,0.578944,0.886733,0.886733,2
1,DBOY1,R1501J,0.471908,0.471908,0.901299,0.901299,2
2,DBOY1,R1502D,0.427843,0.427843,0.775408,0.775408,2
3,DBOY1,R1503E,0.485027,0.485027,0.871999,0.871999,2
4,DBOY1,R1504E,0.463353,0.463353,0.830381,0.830381,1
5,DBOY1,R1509E,0.536898,0.536898,0.733441,0.733441,4
6,DBOY1,R1520T,0.516667,0.516667,0.500000,0.500000,1
7,DBOY1,R1521E,0.494183,0.494183,0.797050,0.797050,3
8,DBOY1,R1523E,0.545125,0.545125,0.838915,0.838915,2
9,DBOY1,R1524T,0.567740,0.567740,0.692963,0.692963,3


In [107]:
import pandas as pd
import numpy as np
import os

# ============================================================================
# CONFIG
# ============================================================================

# SR_REC_WORD clustering output (from clustering_sr_rec_word.py)
SR_MASTER_CSV = './subject_results_clustering_sr_rec_word/ALL_SUBJECTS_clustering_sr_rec_word.csv'

# IRASA pipeline output (from irasa_per_freq pipeline)
IRASA_MASTER_CSV = 'ALL_SUBJECTS_irasa_clean.csv'

OUTPUT_DIR = './subject_results_merged'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# HELPER: two-stage averaging (trial/word → session → subject)
# ============================================================================

def two_stage_avg(df, group_cols_stage1, group_cols_stage2, value_cols):
    """
    Stage 1: average value_cols within group_cols_stage1
    Stage 2: average stage-1 result within group_cols_stage2
    Returns subject-level dataframe.
    """
    stage1 = df.groupby(group_cols_stage1)[value_cols].mean().reset_index()
    stage2 = stage1.groupby(group_cols_stage2)[value_cols].mean().reset_index()
    return stage2

# ============================================================================
# PART 1: SR_REC_WORD clustering → subject-level SP & TC
# ============================================================================

print("Loading SR_REC_WORD clustering data...")
sr_df = pd.read_csv(SR_MASTER_CSV)
print(f"  Rows: {len(sr_df):,}  |  Subjects: {sr_df['subject'].nunique()}")

SR_CLUST_COLS = [
    'SP_clustering_from_prev',
    'SP_clustering_to_next',
    'T_clustering_from_prev',
    'T_clustering_to_next',
]

# Stage 1: average within subject × session (each row is already one recalled word)
# Stage 2: average across sessions within subject
sr_subject_avg = two_stage_avg(
    sr_df,
    group_cols_stage1=['experiment', 'subject', 'session'],
    group_cols_stage2=['experiment', 'subject'],
    value_cols=SR_CLUST_COLS
)

# Rename to make source explicit
sr_subject_avg = sr_subject_avg.rename(columns={
    'SP_clustering_from_prev': 'sr_SP_from_prev',
    'SP_clustering_to_next':   'sr_SP_to_next',
    'T_clustering_from_prev':  'sr_TC_from_prev',
    'T_clustering_to_next':    'sr_TC_to_next',
})

print(f"\nSR subject-level averages ({len(sr_subject_avg)} subjects):")
print(sr_subject_avg.to_string(index=False))

# ============================================================================
# PART 2: IRASA pipeline clustering → subject-level SP & TC
# ============================================================================

print("\nLoading IRASA pipeline data...")
irasa_df = pd.read_csv(IRASA_MASTER_CSV)
print(f"  Rows: {len(irasa_df):,}  |  Subjects: {irasa_df['subject'].nunique()}")

IRASA_CLUST_COLS = [
    'SP_clustering_from_prev',
    'SP_clustering_to_next',
    'T_clustering_from_prev',
    'T_clustering_to_next',
]

# The IRASA CSV has one row per recalled_word × region × freq_hz.
# Clustering scores are the same across region/freq for the same word,
# so collapse to word level first, then session, then subject.
irasa_word_level = (
    irasa_df
    .groupby(['subject', 'session', 'trial', 'recalled_word'])[IRASA_CLUST_COLS]
    .first()   # scores are identical across region×freq rows for same word
    .reset_index()
)

irasa_subject_avg = two_stage_avg(
    irasa_word_level,
    group_cols_stage1=['subject', 'session'],
    group_cols_stage2=['subject'],
    value_cols=IRASA_CLUST_COLS
)

irasa_subject_avg = irasa_subject_avg.rename(columns={
    'SP_clustering_from_prev': 'irasa_SP_from_prev',
    'SP_clustering_to_next':   'irasa_SP_to_next',
    'T_clustering_from_prev':  'irasa_TC_from_prev',
    'T_clustering_to_next':    'irasa_TC_to_next',
})

print(f"\nIRASA subject-level averages ({len(irasa_subject_avg)} subjects):")
print(irasa_subject_avg.to_string(index=False))

# ============================================================================
# PART 3: Merge into one subject-per-row dataframe
# ============================================================================

# SR has 'experiment' column; IRASA is DBOY1 only — add experiment for merge key
irasa_subject_avg['experiment'] = 'DBOY1'

merged = pd.merge(
    sr_subject_avg,
    irasa_subject_avg,
    on=['experiment', 'subject'],
    how='outer'
)

# Reorder columns cleanly
col_order = [
    'experiment', 'subject',
    'sr_SP_from_prev',   'sr_SP_to_next',
    'irasa_SP_from_prev','irasa_SP_to_next',
    'sr_TC_from_prev',   'sr_TC_to_next',
    'irasa_TC_from_prev','irasa_TC_to_next',
]
merged = merged[[c for c in col_order if c in merged.columns]]

print(f"\n{'='*80}")
print(f"Merged subject-level dataframe  ({len(merged)} subjects)")
print(f"{'='*80}")
print(merged.to_string(index=False))

# ============================================================================
# PART 4: Save
# ============================================================================

out_csv = os.path.join(OUTPUT_DIR, 'subject_level_clustering_merged.csv')
merged.to_csv(out_csv, index=False)
print(f"\n✓ Saved: {out_csv}")

# Quick summary stats
print("\n=== Grand means (across subjects) ===")
for col in [c for c in merged.columns if c not in ['experiment', 'subject']]:
    m   = merged[col].mean()
    sem = merged[col].sem()
    n   = merged[col].notna().sum()
    print(f"  {col:<25s}  {m:.4f} ± {sem:.4f}  (N={n})")

Loading SR_REC_WORD clustering data...
  Rows: 845  |  Subjects: 43

SR subject-level averages (43 subjects):
        experiment subject  sr_SP_from_prev  sr_SP_to_next  sr_TC_from_prev  sr_TC_to_next
             DBOY1  R1494D         0.578944       0.578944         0.886733       0.886733
             DBOY1  R1501J         0.471908       0.471908         0.901299       0.901299
             DBOY1  R1502D         0.427843       0.427843         0.775408       0.775408
             DBOY1  R1503E         0.485027       0.485027         0.871999       0.871999
             DBOY1  R1504E         0.463353       0.463353         0.830381       0.830381
             DBOY1  R1509E         0.536898       0.536898         0.733441       0.733441
             DBOY1  R1520T         0.516667       0.516667         0.500000       0.500000
             DBOY1  R1521E         0.494183       0.494183         0.797050       0.797050
             DBOY1  R1523E         0.545125       0.545125         0.83

In [108]:
import pandas as pd
import numpy as np

# ============================================================================
# LOAD
# ============================================================================

merged = pd.read_csv('./subject_results_merged/subject_level_clustering_merged.csv')
print(f"Loaded {len(merged)} subjects")
print(merged[['subject', 'irasa_SP_from_prev', 'irasa_TC_from_prev']].to_string(index=False))

# ============================================================================
# USE from_prev AS PRIMARY CLUSTERING MEASURE (consistent with paper)
# ============================================================================

SP_COL = 'irasa_SP_from_prev'
TC_COL = 'irasa_TC_from_prev'

# Drop rows missing either measure
df = merged.dropna(subset=[SP_COL, TC_COL]).copy()
print(f"\nSubjects with both SP and TC irasa scores: {len(df)}")

# ============================================================================
# DEFINE "HIGH SP, LOW TC" RELATIVE TO SAMPLE MEDIAN
# ============================================================================

sp_median = df[SP_COL].median()
tc_median = df[TC_COL].median()

print(f"\nMedian {SP_COL}: {sp_median:.4f}")
print(f"Median {TC_COL}: {tc_median:.4f}")

high_sp_low_tc = df[
    (df[SP_COL] > sp_median) & (df[TC_COL] < tc_median)
].copy()

high_sp_low_tc = high_sp_low_tc.sort_values(SP_COL, ascending=False).reset_index(drop=True)

print(f"\n{'='*70}")
print(f"Subjects with IRASA SP > median AND IRASA TC < median: {len(high_sp_low_tc)}")
print(f"{'='*70}")
print(high_sp_low_tc.to_string(index=False))

# ============================================================================
# ALSO SHOW SP - TC DIFFERENCE AS A BIAS SCORE
# ============================================================================

df['sp_minus_tc'] = df[SP_COL] - df[TC_COL]
high_sp_low_tc_sorted = (
    df[['experiment', 'subject', SP_COL, TC_COL, 'sp_minus_tc']]
    .sort_values('sp_minus_tc', ascending=False)
    .reset_index(drop=True)
)

print(f"\n{'='*70}")
print("All subjects ranked by SP − TC bias (irasa_SP_from_prev − irasa_TC_from_prev)")
print(f"{'='*70}")
print(high_sp_low_tc_sorted.to_string(index=False))

# Mark the selected subjects
high_sp_low_tc_sorted['selected'] = (
    (high_sp_low_tc_sorted[SP_COL] > sp_median) &
    (high_sp_low_tc_sorted[TC_COL] < tc_median)
)
print(f"\nSelected (SP>median & TC<median): {high_sp_low_tc_sorted['selected'].sum()} subjects")

# ============================================================================
# SAVE
# ============================================================================

out_selected = './subject_results_merged/subjects_high_sp_low_tc.csv'
high_sp_low_tc.to_csv(out_selected, index=False)
print(f"\n✓ Saved selected subjects: {out_selected}")

out_ranked = './subject_results_merged/all_subjects_sp_tc_bias_ranked.csv'
high_sp_low_tc_sorted.to_csv(out_ranked, index=False)
print(f"✓ Saved ranked all subjects: {out_ranked}")

Loaded 47 subjects
subject  irasa_SP_from_prev  irasa_TC_from_prev
 R1494D            0.616296            0.629433
 R1501J            0.513791            0.520526
 R1502D            0.494627            0.637641
 R1503E            0.521559            0.573664
 R1504E            0.526058            0.663267
 R1509E            0.576939            0.596633
 R1520T                 NaN                 NaN
 R1521E            0.464243            0.638233
 R1523E            0.626825            0.831250
 R1524T                 NaN                 NaN
 R1526J                 NaN                 NaN
 R1529T                 NaN                 NaN
 R1535E                 NaN                 NaN
 R1536J            0.499703            0.667341
 R1537T            0.716799            0.691071
 R1538E            0.458102            0.759931
 R1539E                 NaN                 NaN
 R1542J            0.535816            0.558813
 R1543E            0.383190            0.543319
 R1544E              

In [109]:
import pandas as pd
import numpy as np

# ============================================================================
# LOAD
# ============================================================================

merged = pd.read_csv('./subject_results_merged/subject_level_clustering_merged.csv')
print(f"Loaded {len(merged)} subjects")
print(merged[['subject', 'irasa_SP_from_prev', 'irasa_TC_from_prev']].to_string(index=False))

# Drop rows missing any of the four required columns
REQUIRED = ['irasa_SP_from_prev', 'sr_SP_from_prev', 'irasa_TC_from_prev', 'sr_TC_from_prev']
df = merged.dropna(subset=REQUIRED).copy()
print(f"\nSubjects with all four SP/TC scores: {len(df)}")



# ============================================================================
# FILTER: irasa_SP_from_prev >= sr_SP_from_prev
#     AND irasa_TC_from_prev  <  sr_TC_from_prev
# ============================================================================

df['sp_diff'] = df['irasa_SP_from_prev'] - df['sr_SP_from_prev']   # >= 0 means irasa SP higher
df['tc_diff'] = df['irasa_TC_from_prev'] - df['sr_TC_from_prev']   # <  0 means irasa TC lower

selected = df[
    (df['sp_diff'] >= 0) & (df['tc_diff'] < 0)
].copy().reset_index(drop=True)

# Show all subjects with their diffs for context
all_subjects = df[[
    'experiment', 'subject',
    'sr_SP_from_prev',   'irasa_SP_from_prev', 'sp_diff',
    'sr_TC_from_prev',   'irasa_TC_from_prev', 'tc_diff',
]].sort_values('sp_diff', ascending=False).reset_index(drop=True)

all_subjects['selected'] = (
    (all_subjects['sp_diff'] >= 0) & (all_subjects['tc_diff'] < 0)
)

print(f"\n{'='*70}")
print("All subjects — per-subject SP and TC differences (irasa − sr)")
print(f"{'='*70}")
print(all_subjects.to_string(index=False))

print(f"\n{'='*70}")
print(f"Selected: irasa_SP >= sr_SP  AND  irasa_TC < sr_TC  →  {len(selected)} subjects")
print(f"{'='*70}")
print(selected[[
    'experiment', 'subject',
    'sr_SP_from_prev', 'irasa_SP_from_prev', 'sp_diff',
    'sr_TC_from_prev', 'irasa_TC_from_prev', 'tc_diff',
]].to_string(index=False))

# ============================================================================
# SAVE
# ============================================================================

out_all = './subject_results_merged/all_subjects_sp_tc_diff.csv'
all_subjects.to_csv(out_all, index=False)
print(f"\n✓ Saved all subjects with diffs: {out_all}")

out_selected = './subject_results_merged/subjects_irasa_sp_higher_tc_lower.csv'
selected.to_csv(out_selected, index=False)
print(f"✓ Saved selected subjects: {out_selected}")

Loaded 47 subjects
subject  irasa_SP_from_prev  irasa_TC_from_prev
 R1494D            0.616296            0.629433
 R1501J            0.513791            0.520526
 R1502D            0.494627            0.637641
 R1503E            0.521559            0.573664
 R1504E            0.526058            0.663267
 R1509E            0.576939            0.596633
 R1520T                 NaN                 NaN
 R1521E            0.464243            0.638233
 R1523E            0.626825            0.831250
 R1524T                 NaN                 NaN
 R1526J                 NaN                 NaN
 R1529T                 NaN                 NaN
 R1535E                 NaN                 NaN
 R1536J            0.499703            0.667341
 R1537T            0.716799            0.691071
 R1538E            0.458102            0.759931
 R1539E                 NaN                 NaN
 R1542J            0.535816            0.558813
 R1543E            0.383190            0.543319
 R1544E              

In [110]:
"""
LME Analysis: Gamma power ~ source * freq_hz * hemisphere
Restricted to subjects where:
    irasa_SP_from_prev >= sr_SP_from_prev   (IRASA SP >= SR SP)
    irasa_TC_from_prev <  sr_TC_from_prev   (IRASA TC <  SR TC)
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_CSV        = './merged_irasa_rs_recall_vs_retrieval.csv'
SELECTED_SUBJ_CSV = './subject_results_merged/subjects_irasa_sp_higher_tc_lower.csv'
OUTPUT_DIR       = './lme_rs_recall_vs_retrieval_hemi_selected_subj'
os.makedirs(OUTPUT_DIR, exist_ok=True)

POWER_COMPONENTS = ['frac_ssl', 'osc_ssl']
GAMMA_CUTOFF_HZ  = 40.0
REFERENCE_SOURCE = 'RS_Recall'
OTHER_SOURCE     = 'REC_WORD_retrieval'
REFERENCE_HEMI   = 'LHP'
OTHER_HEMI       = 'RHP'

# ============================================================================
# LOAD SELECTED SUBJECTS
# ============================================================================
selected_df = pd.read_csv(SELECTED_SUBJ_CSV)
selected_subjects = set(selected_df['subject'].unique())

print(f"Selected subjects (high IRASA SP, low IRASA TC): {len(selected_subjects)}")
print(sorted(selected_subjects))

# ============================================================================
# LOAD & FILTER
# ============================================================================
print(f"\nLoading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Total rows              : {len(df):,}")

# ── Filter to selected subjects ──────────────────────────────────────────────
df = df[df['subject'].isin(selected_subjects)].copy()
print(f"  After subject filter    : {len(df):,}  ({df['subject'].nunique()} subjects)")

# Correct recalls only
df = df[df['recalled'] == 1].copy()
print(f"  After recalled==1       : {len(df):,}")

# Gamma only
df = df[df['freq_hz'] > GAMMA_CUTOFF_HZ].copy()
print(f"  After freq >40 Hz       : {len(df):,}")
print(f"  Gamma freqs used        : {sorted(df['freq_hz'].unique())}")

# LHP + RHP only
df = df[df['region'].isin([REFERENCE_HEMI, OTHER_HEMI])].copy()

# Keep only subjects with BOTH sources
subjects_rs   = set(df[df['source'] == REFERENCE_SOURCE]['subject'].unique())
subjects_ext  = set(df[df['source'] == OTHER_SOURCE]['subject'].unique())
subjects_both = subjects_rs & subjects_ext

print(f"\n  Subjects with RS_Recall only         : {len(subjects_rs  - subjects_ext)}")
print(f"  Subjects with REC_WORD_retrieval only : {len(subjects_ext - subjects_rs)}")
print(f"  Subjects with BOTH (kept)             : {len(subjects_both)}")

df = df[df['subject'].isin(subjects_both)].copy()
print(f"  Rows after both-source filter         : {len(df):,}")

if len(subjects_both) == 0:
    raise ValueError("No subjects remain after filtering — check that subject IDs match between CSVs.")

# ============================================================================
# DUMMY CODING
# ============================================================================
df['source_code'] = (df['source'] == OTHER_SOURCE).astype(int)
df['hemi_code']   = (df['region'] == OTHER_HEMI).astype(int)
df['subj_sess']   = df['subject'].astype(str) + '_' + df['session'].astype(str)

print(f"\n  source_code distribution : {df['source_code'].value_counts().to_dict()}")
print(f"  hemi_code distribution   : {df['hemi_code'].value_counts().to_dict()}")
print(f"  Final subjects           : {sorted(df['subject'].unique())}")
print(f"  subj_sess groups         : {df['subj_sess'].nunique()}")

# ============================================================================
# LME MODELS
# ============================================================================
all_results = []

for power_col in POWER_COMPONENTS:
    reg_df = df.dropna(subset=[power_col]).copy()

    print(f"\n{'='*70}")
    print(f"  DV: {power_col}  |  N rows: {len(reg_df):,}  |  N subjects: {reg_df['subject'].nunique()}")
    print(f"  Rows per source × hemi:")
    print(reg_df.groupby(['source', 'region']).size().to_string())
    print(f"{'='*70}")

    formula = f'{power_col} ~ source_code * freq_hz * hemi_code'

    try:
        model = smf.mixedlm(
            formula,
            data       = reg_df,
            groups     = reg_df['subject'],
            vc_formula = {'subj_sess': '0 + C(subj_sess)'}
        )
        result = model.fit(reml=True, method='lbfgs', maxiter=1000)
        print(result.summary())

        fe = result.fe_params
        pv = result.pvalues
        ci = result.conf_int()

        for param in fe.index:
            all_results.append({
                'power'       : power_col,
                'parameter'   : param,
                'beta'        : fe[param],
                'pval'        : pv[param],
                'ci_low'      : ci.loc[param, 0],
                'ci_high'     : ci.loc[param, 1],
                'n_rows'      : len(reg_df),
                'n_subjects'  : reg_df['subject'].nunique(),
                'n_subj_sess' : reg_df['subj_sess'].nunique(),
                'converged'   : result.converged,
            })

    except Exception as e:
        print(f"  ✗ Model failed: {e}")
        import traceback; traceback.print_exc()
        continue

# ============================================================================
# RESULTS + FDR CORRECTION
# ============================================================================
results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    print("\n⚠ No results to save.")
else:
    for param in results_df['parameter'].unique():
        mask  = results_df['parameter'] == param
        pvals = results_df.loc[mask, 'pval'].values
        if len(pvals) > 1:
            _, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
            results_df.loc[mask, 'pval_fdr'] = pvals_fdr
        else:
            results_df.loc[mask, 'pval_fdr'] = pvals

    results_df['sig_fdr'] = results_df['pval_fdr'] < 0.05
    results_df['sig_raw'] = results_df['pval']      < 0.05

    results_csv = os.path.join(OUTPUT_DIR, 'lme_results_gamma_hemi_source_selected.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"\n✓ Results saved: {results_csv}")
    print(results_df.to_string(index=False))

    # ============================================================================
    # FOREST PLOT
    # ============================================================================
    params_to_plot = [
        'source_code',
        'freq_hz',
        'hemi_code',
        'source_code:freq_hz',
        'source_code:hemi_code',
        'freq_hz:hemi_code',
        'source_code:freq_hz:hemi_code',
    ]
    param_labels = {
        'source_code'                  : 'Source\n(verbal vs spatial)',
        'freq_hz'                      : 'Frequency (Hz)',
        'hemi_code'                    : 'Hemisphere\n(RHP vs LHP)',
        'source_code:freq_hz'          : 'Source × Freq',
        'source_code:hemi_code'        : 'Source × Hemisphere',
        'freq_hz:hemi_code'            : 'Freq × Hemisphere',
        'source_code:freq_hz:hemi_code': 'Source × Freq\n× Hemisphere',
    }
    colors = {'frac_ssl': '#4C72B0', 'osc_ssl': '#DD8452'}

    fig, ax = plt.subplots(figsize=(7, 7))
    spacing = 0.25

    for i, param in enumerate(params_to_plot):
        for j, power_col in enumerate(POWER_COMPONENTS):
            offset  = (j - 0.5) * spacing
            sub_row = results_df[
                (results_df['parameter'] == param) &
                (results_df['power']     == power_col)
            ]
            if len(sub_row) == 0:
                continue
            row = sub_row.iloc[0]
            y   = i + offset
            col = colors[power_col]
            marker = '*' if row['sig_fdr'] else ('o' if row['sig_raw'] else 's')

            ax.errorbar(
                x          = row['beta'],
                y          = y,
                xerr       = [[row['beta'] - row['ci_low']],
                               [row['ci_high'] - row['beta']]],
                fmt        = marker,
                color      = col,
                markersize = 10 if row['sig_fdr'] else 7,
                capsize    = 4,
                linewidth  = 1.5,
                label      = power_col if i == 0 else ''
            )

    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(params_to_plot)))
    ax.set_yticklabels(
        [param_labels.get(p, p) for p in params_to_plot], fontsize=10
    )
    ax.set_xlabel('β  (95% CI)  —  raw SSL units', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)

    n_subj = df['subject'].nunique()
    handles = [mpatches.Patch(color=colors[p], label=p) for p in POWER_COMPONENTS]
    handles += [
        plt.Line2D([0],[0], marker='*', color='gray', linestyle='', markersize=10, label='p_FDR < 0.05'),
        plt.Line2D([0],[0], marker='o', color='gray', linestyle='', markersize=7,  label='p_raw < 0.05'),
        plt.Line2D([0],[0], marker='s', color='gray', linestyle='', markersize=7,  label='n.s.'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
               bbox_to_anchor=(0.5, -0.06), frameon=False)

    ax.set_title(
        f'Hippocampal Gamma: Source × Hemisphere\n'
        f'Selected subjects (IRASA SP≥SR SP & IRASA TC<SR TC, N={n_subj})\n'
        f'freq > {GAMMA_CUTOFF_HZ} Hz, LHP=reference, RS_Recall=reference',
        fontsize=10
    )
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, 'forest_plot_gamma_hemi_source_selected.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Forest plot saved: {plot_path}")

    # ============================================================================
    # DESCRIPTIVE SUMMARY
    # ============================================================================
    desc_freq = (
        df.groupby(['source', 'region', 'freq_hz'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
        .reset_index()
    )
    desc_csv = os.path.join(OUTPUT_DIR, 'descriptive_gamma_hemi_source_by_freq_selected.csv')
    desc_freq.to_csv(desc_csv, index=False)
    print(f"✓ Descriptive CSV saved: {desc_csv}")

Selected subjects (high IRASA SP, low IRASA TC): 16
['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1523E', 'R1536J', 'R1537T', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1570T', 'R1571T', 'R1572T']

Loading ./merged_irasa_rs_recall_vs_retrieval.csv ...
  Total rows              : 68,598
  After subject filter    : 36,594  (16 subjects)
  After recalled==1       : 35,082
  After freq >40 Hz       : 11,694
  Gamma freqs used        : [42.44, 52.92, 66.0, 82.31, 102.64, 128.0]

  Subjects with RS_Recall only         : 0
  Subjects with REC_WORD_retrieval only : 0
  Subjects with BOTH (kept)             : 16
  Rows after both-source filter         : 11,694

  source_code distribution : {1: 6018, 0: 5676}
  hemi_code distribution   : {0: 6270, 1: 5424}
  Final subjects           : ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1523E', 'R1536J', 'R1537T', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1570T', 'R1571T', 'R1572T']
  subj_sess groups         : 42

  DV

In [112]:
"""
LME Analysis: Gamma power ~ source_code * freq_hz
Run separately for LHP and RHP, for frac_ssl and osc_ssl.

Model:
    power ~ source_code * freq_hz + (1|subject) + (1|subject:session)

Restricted to selected subjects:
    irasa_SP_from_prev >= sr_SP_from_prev
    irasa_TC_from_prev <  sr_TC_from_prev
"""

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
INPUT_CSV         = './merged_irasa_rs_recall_vs_retrieval.csv'
SELECTED_SUBJ_CSV = './subject_results_merged/subjects_irasa_sp_higher_tc_lower.csv'
OUTPUT_DIR        = './lme_rs_recall_vs_retrieval_per_hemi_selected'
os.makedirs(OUTPUT_DIR, exist_ok=True)

POWER_COMPONENTS = ['frac_ssl', 'osc_ssl']
HEMISPHERES      = ['LHP', 'RHP']
GAMMA_CUTOFF_HZ  = 40.0
REFERENCE_SOURCE = 'RS_Recall'
OTHER_SOURCE     = 'REC_WORD_retrieval'

# ============================================================================
# LOAD SELECTED SUBJECTS
# ============================================================================
selected_df       = pd.read_csv(SELECTED_SUBJ_CSV)
selected_subjects = set(selected_df['subject'].unique())
print(f"Selected subjects: {len(selected_subjects)}")
print(sorted(selected_subjects))

# ============================================================================
# LOAD & FILTER
# ============================================================================
print(f"\nLoading {INPUT_CSV} ...")
df = pd.read_csv(INPUT_CSV)
print(f"  Total rows           : {len(df):,}")

df = df[df['subject'].isin(selected_subjects)].copy()
print(f"  After subject filter : {len(df):,}  ({df['subject'].nunique()} subjects)")

df = df[df['recalled'] == 1].copy()
print(f"  After recalled==1    : {len(df):,}")

df = df[df['freq_hz'] > GAMMA_CUTOFF_HZ].copy()
print(f"  After freq >40 Hz    : {len(df):,}")
print(f"  Gamma freqs          : {sorted(df['freq_hz'].unique())}")

df = df[df['region'].isin(HEMISPHERES)].copy()

# Keep only subjects with BOTH sources
subjects_rs   = set(df[df['source'] == REFERENCE_SOURCE]['subject'].unique())
subjects_ext  = set(df[df['source'] == OTHER_SOURCE]['subject'].unique())
subjects_both = subjects_rs & subjects_ext
df = df[df['subject'].isin(subjects_both)].copy()
print(f"  Subjects with both sources: {len(subjects_both)}")
print(f"  Final rows           : {len(df):,}")

if len(subjects_both) == 0:
    raise ValueError("No subjects remain — check subject ID matching between CSVs.")

# ============================================================================
# DUMMY CODING
# ============================================================================
df['source_code'] = (df['source'] == OTHER_SOURCE).astype(int)
# Build subj_sess as a column (used for nested random effects)
df['subj_sess']   = df['subject'].astype(str) + '_' + df['session'].astype(str)

print(f"\n  source_code : {df['source_code'].value_counts().to_dict()}")
print(f"  subjects    : {sorted(df['subject'].unique())}")
print(f"  subj_sess   : {df['subj_sess'].nunique()}")

# ============================================================================
# HELPER: run one LME and return result rows
# ============================================================================
def run_lme(reg_df, power_col, hemi):
    """
    Fit: power ~ source_code * freq_hz
    Random intercepts: subject (groups), session-within-subject (exog_vc).
    Returns list of result dicts, one per fixed-effect parameter.
    """
    # Build session-within-subject design matrix for variance component
    # This is the correct approach for older statsmodels that don't support
    # vc_formula referencing a column by name in the formula string.
    subj_sess_dummies = pd.get_dummies(reg_df['subj_sess'], drop_first=False).astype(float)

    formula = f'{power_col} ~ source_code * freq_hz'

    model = smf.mixedlm(
        formula,
        data      = reg_df,
        groups    = reg_df['subject'],
        exog_vc   = subj_sess_dummies,
    )
    result = model.fit(reml=True, method='lbfgs', maxiter=1000)
    print(result.summary())

    fe = result.fe_params
    pv = result.pvalues
    ci = result.conf_int()

    rows = []
    for param in fe.index:
        rows.append({
            'hemisphere'  : hemi,
            'power'       : power_col,
            'parameter'   : param,
            'beta'        : fe[param],
            'pval'        : pv[param],
            'ci_low'      : ci.loc[param, 0],
            'ci_high'     : ci.loc[param, 1],
            'n_rows'      : len(reg_df),
            'n_subjects'  : reg_df['subject'].nunique(),
            'n_subj_sess' : reg_df['subj_sess'].nunique(),
            'converged'   : result.converged,
        })
    return rows

# ============================================================================
# RUN MODELS: LHP then RHP, frac_ssl then osc_ssl
# ============================================================================
all_results = []

for hemi in HEMISPHERES:
    hemi_df = df[df['region'] == hemi].copy()
    print(f"\n{'='*70}")
    print(f"HEMISPHERE: {hemi}  |  N rows: {len(hemi_df):,}  |  N subjects: {hemi_df['subject'].nunique()}")
    print(hemi_df.groupby('source').size().to_string())
    print(f"{'='*70}")

    for power_col in POWER_COMPONENTS:
        reg_df = hemi_df.dropna(subset=[power_col]).copy()
        print(f"\n  {hemi} | {power_col} | N rows: {len(reg_df):,}")

        try:
            rows = run_lme(reg_df, power_col, hemi)
            all_results.extend(rows)
        except Exception as e:
            print(f"  ✗ {hemi} {power_col} failed: {e}")
            import traceback; traceback.print_exc()

# ============================================================================
# FDR CORRECTION & SAVE
# ============================================================================
results_df = pd.DataFrame(all_results)

if len(results_df) == 0:
    print("\n⚠ No results to save.")
else:
    # FDR within each hemisphere × parameter combination
    for hemi in HEMISPHERES:
        for param in results_df['parameter'].unique():
            mask  = (results_df['hemisphere'] == hemi) & (results_df['parameter'] == param)
            pvals = results_df.loc[mask, 'pval'].values
            if len(pvals) > 1:
                _, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
                results_df.loc[mask, 'pval_fdr'] = pvals_fdr
            else:
                results_df.loc[mask, 'pval_fdr'] = pvals

    results_df['sig_fdr'] = results_df['pval_fdr'] < 0.05
    results_df['sig_raw'] = results_df['pval']      < 0.05

    results_csv = os.path.join(OUTPUT_DIR, 'lme_results_per_hemi.csv')
    results_df.to_csv(results_csv, index=False)
    print(f"\n✓ Results saved: {results_csv}")
    print(results_df.to_string(index=False))

    # ============================================================================
    # FOREST PLOTS — one per hemisphere
    # ============================================================================
    params_to_plot = ['source_code', 'freq_hz', 'source_code:freq_hz']
    param_labels   = {
        'source_code'         : 'Source\n(verbal vs spatial)',
        'freq_hz'             : 'Frequency (Hz)',
        'source_code:freq_hz' : 'Source × Freq',
    }
    colors  = {'frac_ssl': '#4C72B0', 'osc_ssl': '#DD8452'}
    n_subj  = df['subject'].nunique()
    spacing = 0.25

    for hemi in HEMISPHERES:
        hemi_res = results_df[results_df['hemisphere'] == hemi]
        fig, ax  = plt.subplots(figsize=(6, 5))

        for i, param in enumerate(params_to_plot):
            for j, power_col in enumerate(POWER_COMPONENTS):
                offset  = (j - 0.5) * spacing
                sub_row = hemi_res[
                    (hemi_res['parameter'] == param) &
                    (hemi_res['power']     == power_col)
                ]
                if len(sub_row) == 0:
                    continue
                row    = sub_row.iloc[0]
                y      = i + offset
                col    = colors[power_col]
                marker = '*' if row['sig_fdr'] else ('o' if row['sig_raw'] else 's')

                ax.errorbar(
                    x          = row['beta'],
                    y          = y,
                    xerr       = [[row['beta'] - row['ci_low']],
                                  [row['ci_high'] - row['beta']]],
                    fmt        = marker,
                    color      = col,
                    markersize = 10 if row['sig_fdr'] else 7,
                    capsize    = 4,
                    linewidth  = 1.5,
                )

        ax.axvline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_yticks(range(len(params_to_plot)))
        ax.set_yticklabels([param_labels[p] for p in params_to_plot], fontsize=10)
        ax.set_xlabel('β  (95% CI)', fontsize=11)
        ax.spines[['top', 'right']].set_visible(False)

        handles  = [mpatches.Patch(color=colors[p], label=p) for p in POWER_COMPONENTS]
        handles += [
            plt.Line2D([0],[0], marker='*', color='gray', linestyle='', markersize=10, label='p_FDR<0.05'),
            plt.Line2D([0],[0], marker='o', color='gray', linestyle='', markersize=7,  label='p_raw<0.05'),
            plt.Line2D([0],[0], marker='s', color='gray', linestyle='', markersize=7,  label='n.s.'),
        ]
        fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9,
                   bbox_to_anchor=(0.5, -0.08), frameon=False)

        ax.set_title(
            f'{hemi} — Gamma Power: Source × Freq\n'
            f'Selected subjects (N={n_subj}), freq>{GAMMA_CUTOFF_HZ} Hz',
            fontsize=10
        )
        plt.tight_layout()
        plot_path = os.path.join(OUTPUT_DIR, f'forest_plot_{hemi}.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"✓ Forest plot saved: {plot_path}")

    # ============================================================================
    # DESCRIPTIVE SUMMARY
    # ============================================================================
    desc_csv = os.path.join(OUTPUT_DIR, 'descriptive_per_hemi_source_freq.csv')
    desc = (
        df.groupby(['region', 'source', 'freq_hz'])[POWER_COMPONENTS]
        .agg(['mean', 'sem'])
        .round(4)
        .reset_index()
    )
    desc.to_csv(desc_csv, index=False)
    print(f"✓ Descriptive CSV saved: {desc_csv}")

Selected subjects: 16
['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1523E', 'R1536J', 'R1537T', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1570T', 'R1571T', 'R1572T']

Loading ./merged_irasa_rs_recall_vs_retrieval.csv ...
  Total rows           : 68,598
  After subject filter : 36,594  (16 subjects)
  After recalled==1    : 35,082
  After freq >40 Hz    : 11,694
  Gamma freqs          : [42.44, 52.92, 66.0, 82.31, 102.64, 128.0]
  Subjects with both sources: 16
  Final rows           : 11,694

  source_code : {1: 6018, 0: 5676}
  subjects    : ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1523E', 'R1536J', 'R1537T', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1570T', 'R1571T', 'R1572T']
  subj_sess   : 42

HEMISPHERE: LHP  |  N rows: 6,270  |  N subjects: 16
source
REC_WORD_retrieval    3432
RS_Recall             2838

  LHP | frac_ssl | N rows: 5,826
             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:   

Traceback (most recent call last):
  File "<ipython-input-112-d4b406f1b1f2>", line 149, in <module>
    rows = run_lme(reg_df, power_col, hemi)
  File "<ipython-input-112-d4b406f1b1f2>", line 108, in run_lme
    result = model.fit(reml=True, method='lbfgs', maxiter=1000)
  File "/home1/rcao92/.conda/envs/workshop_copy/lib/python3.7/site-packages/statsmodels/regression/mixed_linear_model.py", line 2106, in fit
    **kwargs)
  File "/home1/rcao92/.conda/envs/workshop_copy/lib/python3.7/site-packages/statsmodels/base/model.py", line 526, in fit
    full_output=full_output)
  File "/home1/rcao92/.conda/envs/workshop_copy/lib/python3.7/site-packages/statsmodels/base/optimizer.py", line 218, in _fit
    hess=hessian)
  File "/home1/rcao92/.conda/envs/workshop_copy/lib/python3.7/site-packages/statsmodels/base/optimizer.py", line 440, in _fit_lbfgs
    **extra_kwargs)
  File "/home1/rcao92/.conda/envs/workshop_copy/lib/python3.7/site-packages/scipy/optimize/lbfgsb.py", line 198, in fmin_l_bfgs


✓ Results saved: ./lme_rs_recall_vs_retrieval_per_hemi_selected/lme_results_per_hemi.csv
hemisphere    power           parameter      beta         pval    ci_low   ci_high  n_rows  n_subjects  n_subj_sess  converged     pval_fdr  sig_fdr  sig_raw
       LHP frac_ssl           Intercept  2.064992 1.906259e-23  1.659361  2.470623    5826          12           33       True 3.812517e-23     True     True
       LHP frac_ssl         source_code -0.104751 3.522147e-09 -0.139518 -0.069983    5826          12           33       True 7.044293e-09     True     True
       LHP frac_ssl             freq_hz -0.012394 0.000000e+00 -0.012710 -0.012078    5826          12           33       True 0.000000e+00     True     True
       LHP frac_ssl source_code:freq_hz  0.001402 2.341406e-11  0.000991  0.001814    5826          12           33       True 4.682812e-11     True     True
       LHP  osc_ssl           Intercept  0.652845 2.033589e-12  0.470887  0.834802    5826          12           33     